# Analityka polskich obligacji skarbowych

Dzienny raport portfela długu Skarbu Państwa: outstanding, duration, aukcje, koszt obsługi.
Każda sekcja zawiera wykres, opis kontekstowy oraz komentarz analityczny (Claude Opus 4.8),
plus finalny raport syntezujący wszystkie metryki na końcu.

**Spis treści:**

*Strukturalny snapshot:*
1. Portfolio-weighted metryki (Mod/Mac Duration, ATM, ATR)
2. Per coupon bucket (I/OS/Z)
3. Skład długu — stacked area
4. Skład długu — udział procentowy per bucket
5. Maturity ladder (refinancing profile)

*Aukcje:*
6. Bid/cover + concession w czasie
7. Per coupon bucket
8. Popyt/podaż/wykonanie
9. Tabela ostatniej aukcji
10. Historyczna dystrybucja per seria (box plots)
11. Wpływ aukcji na portfolio
12. Tail / concession
13. Auction scorecard
14. Concession & demand scatters
15. NK (non-competitive) bidding

*Makro:*
16. Funding pace tracker — YTD vs MF plan
17. 🤖 Raport finalny analityczny


In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

load_dotenv(Path("..") / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000

START_DATE = pd.Timestamp("2012-01-01")
START_DATE_STR = START_DATE.strftime("%Y-%m-%d")
TODAY_STR = pd.Timestamp.today().normalize().strftime("%Y-%m-%d")


def _paginate(method, url, *, json_body=None, timeout=120, max_pages=200):
    rows: list = []
    for page in range(max_pages):
        offset = page * PAGE_SIZE
        h = {**HEADERS,
             "Range-Unit": "items",
             "Range": f"{offset}-{offset + PAGE_SIZE - 1}"}
        kw = {"headers": h, "timeout": timeout}
        if json_body is not None:
            kw["json"] = json_body
        r = requests.request(method, url, **kw)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        if not isinstance(chunk, list):
            return chunk
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            return rows
    raise RuntimeError(f"_paginate hit max_pages={max_pages} without finishing")


def rpc(name: str, payload: dict | None = None) -> pd.DataFrame:
    rows = _paginate("POST", f"{SUPABASE_URL}/rest/v1/rpc/{name}",
                     json_body=payload or {})
    return pd.DataFrame(rows)


def fetch_view(name: str, query: str = "?select=*") -> pd.DataFrame:
    rows = _paginate("GET", f"{SUPABASE_URL}/rest/v1/{name}{query}")
    return pd.DataFrame(rows)


# ============================ LLM commentary ============================
# Wymaga ANTHROPIC_API_KEY w env. Kazda odpowiedz zapisywana do tabeli
# llm_commentary (z llm_commentary_schema.sql). Przy kolejnym renderze
# wstrzykujemy HISTORY_N ostatnich analiz tej sekcji do promptu - model
# moze referowac wlasne wczesniejsze myslenie i widziec ciaglosc narracji.

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
LLM_MODEL = "claude-opus-4-8"
HISTORY_N = 3

_llm_client = None
def _get_llm_client():
    global _llm_client
    if _llm_client is not None:
        return _llm_client
    if not ANTHROPIC_API_KEY:
        return None
    try:
        import anthropic
        _llm_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        return _llm_client
    except ImportError:
        return None


_LLM_SYSTEM_PROMPT = (
    "Jesteś doświadczonym analitykiem rynku polskich obligacji skarbowych "
    "(POLGB) pracującym dla sell-side desk. Piszesz precyzyjnie, profesjonalnie, "
    "bez bullshitu, w języku polskim. Używasz terminologii branżowej "
    "(bid-to-cover, concession, through, tail, NK, ATM, ATR, WAM, "
    "PolStr/WIBOR/POLGB itd.). Nie powtarzasz liczb które już są na wykresie - "
    "skupiasz się na interpretacji DANYCH przekazanych w prompcie i mechaniki "
    "aukcyjnej (znaczenie B/C, concession, tail, NK, struktury popytu).\n\n"
    "TWARDE OGRANICZENIA - NIE WOLNO CI:\n"
    "1. Spekulować o oczekiwaniach stóp procentowych (cięcia/podwyżki NBP, EBC, Fed) "
    "ani twierdzić jak rynek się pozycjonuje co do polityki monetarnej - chyba że "
    "konkretne dane (forwards, OIS, ankiety) są w prompcie.\n"
    "2. Cytować/zakładać poziomów inflacji, CPI, PKB, deficytu budżetowego, "
    "potrzeb pożyczkowych itp. których nie ma w danych wejściowych.\n"
    "3. Odwoływać się do sytuacji geopolitycznej, wyborów, ratingu, news flow, "
    "wypowiedzi członków RPP - nie masz tych informacji.\n"
    "4. Używać sformułowań typu 'w środowisku oczekiwań na X', 'rynek wycenia Y', "
    "'sentyment jest Z' bez konkretnego źródła w danych.\n\n"
    "CO WOLNO: interpretować to co widać w danych (popyt vs podaż, koncesje, "
    "preferencje per bucket kuponowy, duration/WAM portfela, trendy serii czasowej), "
    "tłumaczyć mechanikę (np. 'wysoka NK = MF zostawił sobie zapas na popyt wtórny'), "
    "oraz odwoływać się do swoich wcześniejszych analiz tej sekcji jeśli zostały "
    "podane ('jak wskazywałem ostatnio', 'trend kontynuuje się...'). Jeśli dane "
    "nie wystarczają do wniosku - napisz to wprost zamiast zmyślać kontekst."
)


def _fetch_commentary_history(section: str, limit: int = HISTORY_N) -> list:
    """Last N historical commentaries for given section. Returns [] on error."""
    try:
        url = f"{SUPABASE_URL}/rest/v1/rpc/llm_commentary_history"
        r = requests.post(url, headers=HEADERS,
                          json={"p_section": section, "p_limit": limit},
                          timeout=15)
        if r.status_code != 200:
            return []
        data = r.json()
        return data if isinstance(data, list) else []
    except Exception:
        return []


def _save_commentary(section, chart_name, prompt, response,
                      input_tokens, output_tokens):
    """Insert response to llm_commentary table. Silent-fail on error."""
    try:
        url = f"{SUPABASE_URL}/rest/v1/llm_commentary"
        payload = {
            "snapshot_date": TODAY_STR,
            "section": section,
            "chart_name": chart_name,
            "model": LLM_MODEL,
            "prompt": prompt,
            "response": response,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
        }
        r = requests.post(url, headers={**HEADERS, "Prefer": "return=minimal"},
                          json=payload, timeout=20)
        if r.status_code >= 400:
            print(f"  ! save_commentary HTTP {r.status_code}: {r.text[:200]}", flush=True)
    except Exception as exc:
        print(f"  ! save_commentary error: {exc}", flush=True)


def _build_history_block(section: str) -> str:
    """Format last N commentaries as text injected into next prompt."""
    hist = _fetch_commentary_history(section)
    if not hist:
        return ""
    lines = ["", "=== POPRZEDNIE TWOJE ANALIZY TEJ SEKCJI (od najnowszej) ==="]
    for h in hist:
        lines.append(f"\n[{h.get('snapshot_date', '?')}]\n{h.get('response', '')}")
    lines.append("=== END HISTORII ===\n")
    return "\n".join(lines)


def llm_chart_commentary(chart_name: str, what_it_shows: str, data_summary: str,
                          max_tokens: int = 350) -> str:
    """Krotki 2-4 zdaniowy komentarz pod wykresem."""
    client = _get_llm_client()
    if client is None:
        return ("> ⚠️ _Brak ANTHROPIC_API_KEY lub `anthropic` package - "
                "LLM commentary wyłączony._")
    section = chart_name
    history_block = _build_history_block(section)
    prompt = (
        f"WYKRES: **{chart_name}**\n\n"
        f"CO POKAZUJE: {what_it_shows}\n\n"
        f"DANE TERAZ:\n{data_summary}\n"
        + history_block +
        "\nNapisz **krótki** (2-4 zdania) komentarz analityczny w języku polskim. "
        "Bez bullet points, bez powtarzania surowych liczb - skup się na "
        "interpretacji, kontekście historycznym i sygnale rynkowym."
    )
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=max_tokens,
            system=_LLM_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        _save_commentary(
            section=section, chart_name=chart_name, prompt=prompt, response=text,
            input_tokens=getattr(response.usage, "input_tokens", None),
            output_tokens=getattr(response.usage, "output_tokens", None),
        )
        return f"> 💬 **Analiza:** {text}"
    except Exception as exc:
        return f"> ⚠️ _LLM error: {exc}_"


def llm_final_report(context: str, max_tokens: int = 900) -> str:
    """Finalny raport per aukcja."""
    client = _get_llm_client()
    if client is None:
        return ("⚠️ _Brak ANTHROPIC_API_KEY lub `anthropic` package - "
                "finalny raport wyłączony._")
    section = "final_auction_report"
    history_block = _build_history_block(section)
    prompt = (
        "Otrzymujesz pelny kontekst dashboard'u + dzisiejszej aukcji POLGB. "
        "Napisz profesjonalny raport koncowy (250-350 slow) w jezyku polskim.\n\n"
        "**Wymagana struktura:**\n"
        "1. **TL;DR** (1-2 linie: weak/typowo/strong + glowne uzasadnienie)\n"
        "2. **Demand** - co napedzilo/utrudnilo popyt, vs historia per seria\n"
        "3. **Pricing** - concession analysis (through/tail), market context\n"
        "4. **Portfolio impact** - co aukcja zrobila z portfelem\n"
        "5. **Macro/funding context** - gdzie MF jest na rocznym planie, "
        "wnioski dla nastepnych tygodni\n\n"
        f"KONTEKST DZISIAJ:\n{context}\n"
        + history_block +
        "\nRaport (format markdown z naglowkami sekcji **bold**):"
    )
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=max_tokens,
            system=_LLM_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        _save_commentary(
            section=section, chart_name="Final auction report",
            prompt=prompt, response=text,
            input_tokens=getattr(response.usage, "input_tokens", None),
            output_tokens=getattr(response.usage, "output_tokens", None),
        )
        return text
    except Exception as exc:
        return f"⚠️ _LLM error: {exc}_"


print(f"Connected. START_DATE = {START_DATE_STR}")
print(f"LLM commentary: {'enabled (' + LLM_MODEL + ', history N=' + str(HISTORY_N) + ')' if _get_llm_client() else 'disabled (no API key)'}")

# ============================ Macro context ============================
# Pobiera najnowszy snapshot benchmark yieldow / FX / WIBOR / POLSTR / CPI
# + 30d deltas + funding pace YTD. Zwraca sformatowany blok do wstrzykniecia
# w prompt finalnego raportu - LLM widzi RZECZYWISTE poziomy zamiast
# zgadywac z trainingu (zob. system prompt - zakaz halucynacji macro).

def fetch_macro_snapshot():
    """Latest macro values + 30d delta per series + YTD funding pace.
    Returns dict (snap/deltas_30d/pace) or {} on any error."""
    try:
        snap_df = fetch_view("v_macro_snapshot_latest")
        snap = snap_df.iloc[0].to_dict() if len(snap_df) else {}

        hist_df = fetch_view("v_macro_history_30d")
        deltas = {}
        if len(hist_df):
            hist_df = hist_df.copy()
            hist_df["rate_date"] = pd.to_datetime(hist_df["rate_date"])
            for series in hist_df["series"].unique():
                sub = hist_df[hist_df["series"] == series].sort_values("rate_date")
                if len(sub) >= 2:
                    latest = float(sub.iloc[-1]["value_pct"])
                    oldest = float(sub.iloc[0]["value_pct"])
                    deltas[series] = (latest - oldest, sub.iloc[0]["rate_date"].date(),
                                      sub.iloc[-1]["rate_date"].date())

        # PLN swap curve - jeden snapshot z bluegamma. Konwersja do listy
        # dictow per tenor (sorted po tenor_months).
        swap_df = fetch_view("v_swap_curve_pln_latest")
        swap_curve = []
        swap_date = None
        if len(swap_df):
            swap_df = swap_df.sort_values("tenor_months")
            for _, row in swap_df.iterrows():
                swap_curve.append({
                    "tenor": row["tenor"],
                    "months": int(row["tenor_months"]),
                    "rate": float(row["rate_pct"]),
                    "change_pp": float(row["change_pp"]) if row.get("change_pp") is not None else None,
                })
            swap_date = swap_df.iloc[0]["rate_date"]

        return {"snap": snap, "deltas_30d": deltas,
                "swap_curve": swap_curve, "swap_date": swap_date}
    except Exception as exc:
        print(f"[macro] fetch failed: {exc}")
        return {}


def _fmt_pct(v, decimals=2):
    if v is None: return None
    try: return f"{float(v):.{decimals}f}%"
    except (ValueError, TypeError): return None


def _fmt_fx(v, decimals=3):
    if v is None: return None
    try: return f"{float(v):.{decimals}f}"
    except (ValueError, TypeError): return None


def format_macro_block(ctx):
    """Format macro context as compact prompt block. Empty string if no data.
    Output ma byc faktualny - tylko liczby z DB, zero spekulacji."""
    if not ctx: return ""
    snap = ctx.get("snap") or {}
    deltas = ctx.get("deltas_30d") or {}
    if not snap: return ""

    lines = ["=== KONTEKST MAKRO (faktyczne dane z DB - bez spekulacji) ==="]

    pl10y = snap.get("pl10y"); de10y = snap.get("de10y"); us10y = snap.get("us10y")
    if pl10y is not None:
        pl_line = f"PL10Y={float(pl10y):.2f}%"
        if "PL10Y" in deltas:
            d, d0, d1 = deltas["PL10Y"]
            pl_line += f" (zmiana od {d0}: {d*100:+.0f}bp)"
        if de10y is not None:
            spread = (float(pl10y) - float(de10y)) * 100
            pl_line += f", DE10Y={float(de10y):.2f}%, spread PL-DE = +{spread:.0f}bp"
        if us10y is not None:
            pl_line += f", US10Y={float(us10y):.2f}%"
        lines.append(pl_line)

    wibor = snap.get("wibor6m"); polstr = snap.get("polstr"); cpi = snap.get("cpi_yoy")
    rate_parts = []
    if wibor is not None:
        s = f"WIBOR6M={float(wibor):.2f}%"
        if "WIBOR6M" in deltas:
            d, d0, _ = deltas["WIBOR6M"]
            s += f" (Δ30d: {d*100:+.0f}bp)"
        rate_parts.append(s)
    if polstr is not None:
        s = f"POLSTR={float(polstr):.2f}%"
        if "POLSTR" in deltas:
            d, _, _ = deltas["POLSTR"]
            s += f" (Δ30d: {d*100:+.0f}bp)"
        rate_parts.append(s)
    if cpi is not None: rate_parts.append(f"CPI YoY={float(cpi):.1f}%")
    if rate_parts: lines.append(", ".join(rate_parts))

    fx_parts = []
    eurpln = snap.get("eurpln"); usdpln = snap.get("usdpln"); eurusd = snap.get("eurusd")
    if eurpln is not None:
        s = f"EUR/PLN={float(eurpln):.3f}"
        if "EURPLN" in deltas:
            d, _, _ = deltas["EURPLN"]
            s += f" (Δ30d: {d:+.3f})"
        fx_parts.append(s)
    if usdpln is not None: fx_parts.append(f"USD/PLN={float(usdpln):.3f}")
    if eurusd is not None: fx_parts.append(f"EUR/USD={float(eurusd):.4f}")
    if fx_parts: lines.append(", ".join(fx_parts))

    # NOTE: funding pace celowo NIE injektujemy do LLM context.
    # v_funding_pace_ytd istnieje ale wymaga reczego wpisu annual plan w
    # funding_plan_annual - bedzie zautomatyzowane pozniej (scraper MF).

    # PLN swap curve - kluczowy sygnal market-implied rate path.
    # Pokazujemy 6M, 1Y, 2Y, 5Y, 10Y, 20Y (skipujemy granular tenorów zeby
    # nie zalewac promptu) + Δ1d na 5Y jako proxy daily move.
    swap_curve = ctx.get("swap_curve") or []
    swap_date = ctx.get("swap_date")
    if swap_curve:
        key_tenors = {"6M", "1Y", "2Y", "5Y", "10Y", "20Y"}
        key_points = [c for c in swap_curve if c["tenor"] in key_tenors]
        if key_points:
            curve_str = ", ".join(f"{c['tenor']}={c['rate']:.2f}%" for c in key_points)
            line = f"PLN IRS swap curve ({swap_date}): {curve_str}"
            # Slope hints (model dostaje surowe liczby, slope to interpretacja)
            d_5y = next((c for c in swap_curve if c["tenor"] == "5Y"), None)
            if d_5y and d_5y.get("change_pp") is not None:
                line += f" | Δ1d na 5Y: {d_5y['change_pp']:+.2f}pp"
            lines.append(line)
            # Front-end slope (sygnal expectations)
            r_6m = next((c["rate"] for c in swap_curve if c["tenor"] == "6M"), None)
            r_1y = next((c["rate"] for c in swap_curve if c["tenor"] == "1Y"), None)
            r_2y = next((c["rate"] for c in swap_curve if c["tenor"] == "2Y"), None)
            if r_6m is not None and r_1y is not None:
                slope_6m_1y = (r_1y - r_6m) * 100
                hint = ""
                if abs(slope_6m_1y) < 10:
                    hint = " (flat = market wycenia rates stable)"
                elif slope_6m_1y < 0:
                    hint = " (inverted = market wycenia cuts)"
                else:
                    hint = " (upward = market wycenia hold/hikes)"
                line2 = f"  Slope 6M->1Y: {slope_6m_1y:+.0f}bp{hint}"
                if r_2y is not None and r_1y is not None:
                    line2 += f", 1Y->2Y: {(r_2y - r_1y)*100:+.0f}bp"
                lines.append(line2)

    lines.append("")  # blank line separator
    return "\n".join(lines) + "\n"


## 1. Portfolio-weighted metryki w czasie

Z `v_portfolio_metrics_daily` (ważone outstanding-em dziennym, bondy hurtowe).

In [ ]:
df1 = fetch_view(
    "v_portfolio_metrics_daily",
    f"?fixing_date=gte.{START_DATE_STR}&select=*&order=fixing_date.asc",
)
df1["fixing_date"] = pd.to_datetime(df1["fixing_date"])
for c in ["portfolio_mod_duration", "portfolio_mac_duration", "portfolio_atm",
         "portfolio_atr", "portfolio_yield_pct", "total_outstanding_mln_pln"]:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

print(f"Days: {len(df1)},  range: {df1.fixing_date.min().date()} → {df1.fixing_date.max().date()}")
df1.tail()

In [ ]:
fig = go.Figure()
for col, label in [
    ("portfolio_mod_duration", "Modified Duration"),
    ("portfolio_mac_duration", "Macaulay Duration"),
    ("portfolio_atm", "ATM (years to maturity)"),
    ("portfolio_atr", "ATR (years to refixing)"),
]:
    fig.add_trace(go.Scatter(x=df1["fixing_date"], y=df1[col], name=label, mode="lines"))

fig.update_layout(
    title="Portfolio-weighted metryki polskiego długu (bondy hurtowe)",
    xaxis_title="Data fixingu (EOD = sesja 2)",
    yaxis_title="Lata",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_latest_1 = df1.iloc[-1]
_3mo_1 = df1[df1.fixing_date >= df1.fixing_date.max() - pd.Timedelta(days=90)]
_summary_1 = f"""Latest snapshot ({_latest_1.fixing_date.date()}):
- Modified Duration: {_latest_1.portfolio_mod_duration:.2f}Y
- Macaulay Duration: {_latest_1.portfolio_mac_duration:.2f}Y
- ATM: {_latest_1.portfolio_atm:.2f}Y
- ATR: {_latest_1.portfolio_atr:.2f}Y
- Total outstanding: {_latest_1.total_outstanding_mln_pln/1000:.0f} bln PLN

Last 3 months trend:
- Mod Duration range: {_3mo_1.portfolio_mod_duration.min():.2f} - {_3mo_1.portfolio_mod_duration.max():.2f}
- ATM range: {_3mo_1.portfolio_atm.min():.2f} - {_3mo_1.portfolio_atm.max():.2f}

Full history range (od 2012):
- Mod Duration: {df1.portfolio_mod_duration.min():.2f} - {df1.portfolio_mod_duration.max():.2f}
- ATM: {df1.portfolio_atm.min():.2f} - {df1.portfolio_atm.max():.2f}"""
display(Markdown(llm_chart_commentary(
    "Chart 1 — Portfolio-weighted metryki",
    "Modified Duration, Macaulay Duration, ATM (avg time to maturity) i ATR (avg time to refixing) całego portfela POLGB hurtowych, ważone outstanding. Pokazuje strukturalne ryzyko stopy + maturity profile w czasie.",
    _summary_1,
)))

## 2. Per rodzaj kuponu (I / OS / Z)

Każdy z 4 paneli pokazuje jedną metrykę ważoną outstanding-em, kolory = bucket kuponowy:
- **I** — inflacyjne (IZ)
- **OS** — stałe + zerokuponowe (OK, DS, PS, WS, …)
- **Z** — zmienne (WZ, NZ)

Czego się spodziewać:
- **OS** — duration zależne od mixu krótkich (OK) i długich (WS) — średnio rośnie z trendem wydłużania krzywej
- **Z** (floatery) — Mod/Mac ≈ ATR (≤ 0.5Y), bo resetują kupon co 6mc
- **I** (inflation-linked) — Mod/Mac niskie, ATM długie (długie wykupy ale małe duration bo CPI-link)

In [ ]:
# Bucket mapping wspolny dla chart 2 i 3b (definiujemy raz).
BOND_TYPE_TO_BUCKET = {
    "IZ": "I",
    "WZ": "Z", "NZ": "Z",
}

def to_bucket(bt: str) -> str:
    if bt == "tbill":
        return "tbill"
    return BOND_TYPE_TO_BUCKET.get(bt, "OS")

df2 = fetch_view(
    "v_portfolio_metrics_by_type",
    f"?fixing_date=gte.{START_DATE_STR}&select=*&order=fixing_date.asc,bond_type.asc",
)
df2["fixing_date"] = pd.to_datetime(df2["fixing_date"])
for c in ["total_mln_pln", "w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]:
    df2[c] = pd.to_numeric(df2[c], errors="coerce")

df2["bucket"] = df2["bond_type"].map(to_bucket)

# Re-aggregacja per (fixing_date, bucket): weighted_avg po outstanding.
# Kazda metryka w v_portfolio_metrics_by_type to juz weighted_avg per typ
# (sum(metric_isin * outstanding_isin) / sum(outstanding_isin)). Zeby
# polaczyc kilka typow w bucket bierzemy:
#   bucket_w_metric = sum(w_metric_t * total_t) / sum(total_t)
# co matematycznie odpowiada laczeniu surowych (metric_isin * outstanding_isin)
# z wszystkich typow w buckecie.
METRICS = ["w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]
for m in METRICS:
    # NaN-safe: jesli w_metric jest NaN (np. w_yield_pct dla pewnych dni) to
    # ten kawalek nie wchodzi ani do licznika ani do mianownika.
    df2[f"_num_{m}"] = df2[m] * df2["total_mln_pln"]
    df2[f"_den_{m}"] = df2["total_mln_pln"].where(df2[m].notna())

agg = df2.groupby(["fixing_date", "bucket"], as_index=False).agg(
    total_mln_pln=("total_mln_pln", "sum"),
    **{f"_num_{m}": (f"_num_{m}", "sum") for m in METRICS},
    **{f"_den_{m}": (f"_den_{m}", "sum") for m in METRICS},
)
for m in METRICS:
    agg[m] = agg[f"_num_{m}"] / agg[f"_den_{m}"].replace(0, pd.NA)
    agg.drop(columns=[f"_num_{m}", f"_den_{m}"], inplace=True)

df2 = agg  # od teraz df2 ma kolumny: fixing_date, bucket, total_mln_pln, w_*

print(f"Rows: {len(df2)},  buckets: {sorted(df2.bucket.unique())},  "
      f"range: {df2.fixing_date.min().date()} → {df2.fixing_date.max().date()}")
df2.tail()

In [ ]:
metrics = [
    ("w_mod_duration", "Modified Duration (lata)"),
    ("w_mac_duration", "Macaulay Duration (lata)"),
    ("w_atm", "ATM (lata)"),
    ("w_atr", "ATR (lata)"),
]
fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

# Spojny color scheme z chart 3b
BUCKET_COLORS_2 = {
    "OS": "#1f77b4",  # niebieski
    "Z": "#ff7f0e",   # pomaranczowy
    "I": "#9467bd",   # fioletowy
}
buckets_sorted = [b for b in ["OS", "Z", "I"] if b in df2.bucket.unique()]

for i, (col, _) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    for bucket in buckets_sorted:
        sub = df2[df2.bucket == bucket].sort_values("fixing_date")
        fig.add_trace(
            go.Scatter(
                x=sub["fixing_date"], y=sub[col],
                name=bucket, legendgroup=bucket,
                showlegend=(i == 0), mode="lines",
                line=dict(color=BUCKET_COLORS_2[bucket], width=1.8),
            ),
            row=row, col=c,
        )

fig.update_layout(
    title="Metryki ważone outstanding per rodzaj kuponu (I / OS / Z)",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.05),
)
fig.show()

# LLM commentary
_latest_2 = df2[df2.fixing_date == df2.fixing_date.max()].set_index("bucket")
_lines_2 = []
for bucket in buckets_sorted:
    if bucket in _latest_2.index:
        r = _latest_2.loc[bucket]
        _lines_2.append(
            f"  {bucket}: Mod {r.w_mod_duration:.2f}Y, Mac {r.w_mac_duration:.2f}Y, "
            f"ATM {r.w_atm:.2f}Y, ATR {r.w_atr:.2f}Y, "
            f"outstanding {r.total_mln_pln/1000:.0f} bln"
        )
_summary_2 = "Latest snapshot per bucket:\n" + "\n".join(_lines_2)
display(Markdown(llm_chart_commentary(
    "Chart 2 — Metryki per coupon bucket (I/OS/Z)",
    "Te same 4 metryki co chart 1 ale per coupon bucket: I (inflacyjne), OS (stałe+zerokup), Z (zmienne). Pokazuje jak różne segmenty portfela kontrybuują do całości; floatery typowo mają niskie Mod/Mac, IZ wysokie ATM ale niskie Mod.",
    _summary_2,
)))

## 3. Skład długu — stacked area

Bondy hurtowe per typ + bony skarbowe (jako jeden kubełek `tbill`). Pokazuje jak ewoluowała struktura zadłużenia.

Resample do miesięcznych snapshotów (end-of-month), inaczej wykres jest zaszumiony przy 3000+ dni × 8 typów.

In [ ]:
TODAY = pd.Timestamp.today().normalize()

# Event-driven sklad dlugu z MF (niezalezne od BondSpota - chwytamy NZ-tki
# od dnia pierwszej emisji w MF, nie od pierwszego BondSpotowego fixingu).
# Widoki v_bond_outstanding_by_type_events i v_tbill_outstanding_events
# robia cumulative sum delty per typ na poziomie SQL.

bond_events = fetch_view(
    "v_bond_outstanding_by_type_events",
    "?select=*&order=change_date.asc,bond_type.asc",
)
bond_events["change_date"] = pd.to_datetime(bond_events["change_date"])
bond_events["outstanding_mln_pln"] = pd.to_numeric(
    bond_events["outstanding_mln_pln"], errors="coerce"
)

tbill_events = fetch_view(
    "v_tbill_outstanding_events",
    "?select=*&order=change_date.asc",
)
tbill_events["change_date"] = pd.to_datetime(tbill_events["change_date"])
tbill_events["outstanding_mln_pln"] = pd.to_numeric(
    tbill_events["outstanding_mln_pln"], errors="coerce"
)

# Diagnostyka source: dla kazdego typu pokazujemy ile eventow i ostatnia
# wartosc cumulative (z SQL). Latest_at_today odzwierciedla wartosc po
# odfiltrowaniu przyszlych synthetic redemption events.
print("Source events per bond_type (with cumulative latest <= TODAY):")
diag = bond_events.copy()
diag_today = diag[diag["change_date"] <= TODAY]
src_stats = diag_today.groupby("bond_type").agg(
    n_events=("change_date", "count"),
    first_date=("change_date", "min"),
    last_date=("change_date", "max"),
    last_outstanding=("outstanding_mln_pln", "last"),
).sort_values("last_outstanding", ascending=False)
print(src_stats)
print(f"\ntbill source: {len(tbill_events)} events, "
      f"last <= TODAY: {tbill_events[tbill_events['change_date'] <= TODAY]['outstanding_mln_pln'].iloc[-1]:.1f} "
      f"on {tbill_events[tbill_events['change_date'] <= TODAY]['change_date'].max().date()}")

# Pelny dzienny index od najwczesniejszego eventu do TODAY.
# Reindex+ffill PER typ jest bardziej deterministyczne niz pivot+resample
# (pivot_table moze dla niektorych dtype-ow wstawic 0 zamiast NaN i wtedy
# ffill nie propaguje wartosci - co zlobalismy wczesniej).
data_min = min(bond_events["change_date"].min(), tbill_events["change_date"].min())
all_dates = pd.date_range(start=data_min, end=TODAY, freq="D")

wide = pd.DataFrame(index=all_dates)
for bt in sorted(bond_events["bond_type"].dropna().unique()):
    sub = bond_events[bond_events["bond_type"] == bt].sort_values("change_date")
    s = sub.set_index("change_date")["outstanding_mln_pln"]
    s = s[~s.index.duplicated(keep="last")]
    s = s[s.index <= TODAY]  # bez przyszlych redemption events
    wide[bt] = s.reindex(all_dates).ffill().fillna(0)

ts = tbill_events.set_index("change_date")["outstanding_mln_pln"]
ts = ts[~ts.index.duplicated(keep="last")]
ts = ts[ts.index <= TODAY]
wide["tbill"] = ts.reindex(all_dates).ffill().fillna(0)

# Trim do START_DATE (global z setup-a)
wide = wide[wide.index >= START_DATE]

# Reduce do dni z faktycznymi zmianami (auction-event datapointy zamiast daily)
mask = (wide != wide.shift(1)).any(axis=1)
wide = wide[mask]

# Order: typy od najwiekszego sredniego outstanding
order = wide.mean().sort_values(ascending=False).index.tolist()
wide = wide[order]
piv = wide  # alias dla chart3-plot

print(f"\nEvent dates: {len(piv)}, range: {piv.index.min().date()} → {piv.index.max().date()}")
print(f"Latest total: {piv.iloc[-1].sum() / 1000:.1f} bln PLN")
print(f"Latest per type: {piv.iloc[-1].to_dict()}")
print(f"Types: {order}")
piv.tail()

In [ ]:
fig = go.Figure()
for bt in order:
    fig.add_trace(go.Scatter(
        x=piv.index, y=piv[bt] / 1000.0,
        name=bt, mode="lines", stackgroup="one",
        hovertemplate="%{y:.1f} bln PLN<extra>" + bt + "</extra>",
    ))

# Niewidoczna linia na szczycie stack-u zeby unified hover pokazywal Σ TOTAL.
totals_bln = piv.sum(axis=1) / 1000.0
fig.add_trace(go.Scatter(
    x=piv.index, y=totals_bln,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{y:.1f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="Skład długu skarbowego (bondy hurtowe + bony) — auction events",
    xaxis_title="Data aukcji / odkupu",
    yaxis_title="Outstanding (bln PLN)",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_latest_3 = piv.iloc[-1]
_total_3 = _latest_3.sum() / 1000
_top3 = _latest_3.sort_values(ascending=False).head(3)
_year_ago_3 = piv[piv.index <= piv.index.max() - pd.Timedelta(days=365)]
_yoy = ((_latest_3.sum() / _year_ago_3.iloc[-1].sum()) - 1) * 100 if len(_year_ago_3) else 0
_summary_3 = (
    f"Latest total: {_total_3:.0f} bln PLN ({_latest_3.name.date()})\n"
    f"Top 3 typy: " + ", ".join(f"{t}={v/1000:.0f}bln" for t, v in _top3.items()) + "\n"
    f"YoY change in total: {_yoy:+.1f}%\n"
    f"Liczba typów aktywnych: {(_latest_3 > 0).sum()}"
)
display(Markdown(llm_chart_commentary(
    "Chart 3 — Skład długu (stacked area, per bond_type)",
    "Stacked area showing total outstanding per bond_type (DS, PS, WS, OK, WZ, NZ, IZ, tbill, ...) w czasie. Event-driven (auction dates), nie daily fixings.",
    _summary_3,
)))

## 4. Skład długu — udział procentowy per rodzaj kuponu

Re-bucketing chart 3 z **per bond_type** (DS/WZ/PS/...) na **per coupon kind**:
- **I** — inflacyjne (IZ)
- **OS** — stałe + zerokuponowe (OK, DS, PS, WS, OS, PP, AS, CK, KO, DK, DZ, PK, RP, TK)
- **Z** — zmienne (WZ, NZ)
- **tbill** — bony skarbowe (osobny kubełek)

Wykres jako stacked area znormalizowany do 100% — pokazuje jak ewoluowała **struktura procentowa** zadłużenia (nie nominalna).

In [ ]:
# to_bucket() i BOND_TYPE_TO_BUCKET sa zdefiniowane w chart2-data (reuse).
# Reuse piv (wide DF z chart3-data, juz pelne ffilled per-day + zredukowane
# do event-only rows). Sumujemy kolumny tego samego bucketu.
piv_buckets = pd.DataFrame(index=piv.index)
for bt in piv.columns:
    bucket = to_bucket(bt)
    if bucket not in piv_buckets.columns:
        piv_buckets[bucket] = 0.0
    piv_buckets[bucket] = piv_buckets[bucket] + piv[bt]

# Procenty per row (udzial w calosci emisji na ta date)
row_sums = piv_buckets.sum(axis=1)
piv_pct = piv_buckets.div(row_sums.replace(0, pd.NA), axis=0) * 100
piv_pct = piv_pct.fillna(0)

# Order rysowania: OS na dole (najwiekszy), potem Z, I, tbill na gorze
BUCKET_ORDER = ["OS", "Z", "I", "tbill"]
piv_pct = piv_pct[[c for c in BUCKET_ORDER if c in piv_pct.columns]]
piv_buckets = piv_buckets[piv_pct.columns]

print(f"Buckets: {list(piv_pct.columns)}")
print(f"Latest total: {row_sums.iloc[-1] / 1000:.1f} bln PLN")
print(f"Latest % per bucket: "
      f"{ {k: f'{v:.1f}%' for k, v in piv_pct.iloc[-1].to_dict().items()} }")
print(f"Latest nominal per bucket (mln PLN): "
      f"{ {k: f'{v:,.0f}' for k, v in piv_buckets.iloc[-1].to_dict().items()} }")
piv_pct.tail()

In [ ]:
BUCKET_COLORS = {
    "OS": "#1f77b4",      # niebieski - stale+zerokup (dominanta)
    "Z": "#ff7f0e",       # pomaranczowy - zmienne (WZ/NZ)
    "I": "#9467bd",       # fioletowy - inflacyjne (IZ)
    "tbill": "#7f7f7f",   # szary - bony skarbowe
}

fig = go.Figure()
totals_bln_3b = row_sums / 1000.0  # bln PLN per data (do hover-a)

for bucket in piv_pct.columns:
    nominal_bln = piv_buckets[bucket] / 1000.0
    # customdata = nominal w bln, zeby pokazac obok % w hoverze
    fig.add_trace(go.Scatter(
        x=piv_pct.index, y=piv_pct[bucket],
        customdata=nominal_bln,
        name=bucket, mode="lines", stackgroup="one",
        line=dict(width=0.5, color=BUCKET_COLORS.get(bucket, "grey")),
        hovertemplate=("%{y:.1f}%  (%{customdata:.1f} bln PLN)"
                       "<extra>" + bucket + "</extra>"),
    ))

# Σ TOTAL trace - niewidoczna linia na szczycie (y=100 zeby zawsze byla na
# top edge), w hoverze pokazuje absolutny total w bln PLN.
fig.add_trace(go.Scatter(
    x=piv_pct.index, y=[100.0] * len(piv_pct),
    customdata=totals_bln_3b,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{customdata:.1f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="Skład długu skarbowego — udział procentowy per rodzaj kuponu",
    xaxis_title="Data aukcji / odkupu",
    yaxis_title="Udział (%)",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_latest_3b = piv_pct.iloc[-1]
_yr_ago_3b = piv_pct[piv_pct.index <= piv_pct.index.max() - pd.Timedelta(days=365)]
_pct_lines = [f"  {b}: {_latest_3b[b]:.1f}%" for b in piv_pct.columns]
_shift_lines = []
if len(_yr_ago_3b):
    _prev_3b = _yr_ago_3b.iloc[-1]
    for b in piv_pct.columns:
        delta = _latest_3b[b] - _prev_3b.get(b, 0)
        _shift_lines.append(f"  {b}: {delta:+.1f}pp")
_summary_3b = (
    "Latest mix:\n" + "\n".join(_pct_lines) +
    "\n\nYoY shift in mix:\n" + "\n".join(_shift_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 4 — Struktura % per coupon bucket",
    "Stacked 100% area: udziały coupon buckets (OS=stałe+zerokup, Z=zmienne, I=inflacyjne, tbill=bony) w całym długu. Pokazuje strukturalne tendencje (np. wzrost udziału floaterów w okresie inflacyjnym).",
    _summary_3b,
)))

## 5. Maturity ladder — refinancing profile (Bloomberg DDIS style)

Outstanding obligacji + bonów skarbowych per rok wykupu (jak Bloomberg `DDIS` dla sovereign issuera). Pokazuje **wall of refinancing** — gdzie MF będzie miało skoncentrowane wykupy w nadchodzących latach.

- **Stacked bars**: per maturity year, kolor = coupon bucket (OS/Z/I) + tbill
- **Annotation na bar**: total bln PLN w danym roku
- **Header subtitle**: total outstanding + WAM (weighted avg maturity)
- **Next 12M callout**: ile zapada w ciągu najbliższego roku (short-term refinancing pressure)

Lat z największym total = peak refinancing year — MF musi mieć plan jak to zrolować.

In [ ]:
# Aktywne obligacje + outstanding na TODAY
specs_12 = fetch_view(
    "bond_specs",
    "?select=isin,bond_type,coupon_kind,maturity_date",
)
specs_12["maturity_date"] = pd.to_datetime(specs_12["maturity_date"])

today_str = TODAY.strftime("%Y-%m-%d")
bonds_bal = rpc("bonds_outstanding_at", {"p_date": today_str})
bonds_bal["balance_mln_pln"] = pd.to_numeric(bonds_bal["balance_mln_pln"], errors="coerce")

bonds_lad = specs_12.merge(bonds_bal, on="isin", how="inner")
bonds_lad = bonds_lad[
    (bonds_lad["balance_mln_pln"] > 0)
    & (bonds_lad["maturity_date"] > TODAY)
].copy()
bonds_lad["maturity_year"] = bonds_lad["maturity_date"].dt.year
bonds_lad["bucket"] = bonds_lad["coupon_kind"].map(
    lambda k: "I" if k == "I" else "Z" if k == "Z" else "OS"
)

# Tbills
tbill_specs_12 = fetch_view("tbill_specs", "?select=isin,maturity_date")
tbill_specs_12["maturity_date"] = pd.to_datetime(tbill_specs_12["maturity_date"])
tbill_bal = rpc("tbills_outstanding_at", {"p_date": today_str})
tbill_bal["balance_mln_pln"] = pd.to_numeric(tbill_bal["balance_mln_pln"], errors="coerce")
tbill_lad = tbill_specs_12.merge(tbill_bal, on="isin", how="inner")
tbill_lad = tbill_lad[
    (tbill_lad["balance_mln_pln"] > 0)
    & (tbill_lad["maturity_date"] > TODAY)
].copy()
tbill_lad["maturity_year"] = tbill_lad["maturity_date"].dt.year
tbill_lad["bucket"] = "tbill"

# Polacz dla agregacji
ladder = pd.concat([
    bonds_lad[["maturity_year", "maturity_date", "bucket", "balance_mln_pln"]],
    tbill_lad[["maturity_year", "maturity_date", "bucket", "balance_mln_pln"]],
], ignore_index=True)

# Per (year, bucket) total
agg_12 = ladder.groupby(["maturity_year", "bucket"], as_index=False)["balance_mln_pln"].sum()
pivot_12 = agg_12.pivot(index="maturity_year", columns="bucket", values="balance_mln_pln").fillna(0)
BUCKET_ORDER_12 = ["OS", "Z", "I", "tbill"]
pivot_12 = pivot_12[[b for b in BUCKET_ORDER_12 if b in pivot_12.columns]].sort_index()

# === Matured YTD (bonds + tbills ktore zapadly w tym roku przed TODAY) ===
# Dla pelnego obrazu rocznych wykupow: to co juz zostalo zrolowane + remaining.
year_start_12 = pd.Timestamp(year=TODAY.year, month=1, day=1)
mat_bonds_raw = fetch_view(
    "bond_outstanding",
    f"?op_type=eq.redemption"
    f"&change_date=gte.{year_start_12.strftime('%Y-%m-%d')}"
    f"&change_date=lte.{today_str}"
    "&select=isin,delta_mln_pln",
)
mat_bonds_raw["delta_mln_pln"] = pd.to_numeric(mat_bonds_raw["delta_mln_pln"], errors="coerce")
mat_bonds_raw = mat_bonds_raw.merge(
    specs_12[["isin", "coupon_kind"]], on="isin", how="left"
)
mat_bonds_raw["bucket"] = mat_bonds_raw["coupon_kind"].map(
    lambda k: "I" if k == "I" else "Z" if k == "Z" else "OS"
)
mat_bonds_raw["abs_amt"] = -mat_bonds_raw["delta_mln_pln"]
matured_by_bucket = mat_bonds_raw.groupby("bucket")["abs_amt"].sum()

mat_tbills_raw = fetch_view(
    "tbill_outstanding",
    f"?op_type=eq.redemption"
    f"&change_date=gte.{year_start_12.strftime('%Y-%m-%d')}"
    f"&change_date=lte.{today_str}"
    "&select=delta_mln_pln",
)
mat_tbills_raw["delta_mln_pln"] = pd.to_numeric(mat_tbills_raw["delta_mln_pln"], errors="coerce")
tbill_matured = -mat_tbills_raw["delta_mln_pln"].sum() if len(mat_tbills_raw) else 0.0
if tbill_matured > 0:
    matured_by_bucket["tbill"] = matured_by_bucket.get("tbill", 0) + tbill_matured

matured_ytd_total = matured_by_bucket.sum() if len(matured_by_bucket) else 0.0

# Stats do annotations
yearly_totals = pivot_12.sum(axis=1)
total_outstanding = yearly_totals.sum()
peak_year = yearly_totals.idxmax()
peak_val = yearly_totals.max()

# WAM (weighted avg maturity in years)
ladder["yrs_to_mat"] = (ladder["maturity_date"] - TODAY).dt.days / 365.25
wam_yrs = (ladder["yrs_to_mat"] * ladder["balance_mln_pln"]).sum() / ladder["balance_mln_pln"].sum()

# Next 12M maturities
mat_12m = ladder[ladder["maturity_date"] <= TODAY + pd.Timedelta(days=365)][
    "balance_mln_pln"
].sum()

print(f"Total outstanding ({TODAY.date()}): {total_outstanding/1000:.1f} bln PLN")
print(f"WAM: {wam_yrs:.2f} lat")
print(f"Peak year: {peak_year} ({peak_val/1000:.1f} bln PLN)")
print(f"Next 12M maturities: {mat_12m/1000:.1f} bln PLN ({100*mat_12m/total_outstanding:.1f}% portfela)")
print(f"\nMatured YTD {TODAY.year}: {matured_ytd_total/1000:.1f} bln PLN "
      f"(pokazane na barze {TODAY.year} jako semi-transparent + hatched)")
if len(matured_by_bucket):
    for b, v in matured_by_bucket.items():
        print(f"   {b}: {v/1000:.1f} bln")
print(f"\nFull-year {TODAY.year} maturities (matured + remaining): "
      f"{(matured_ytd_total + yearly_totals.get(TODAY.year, 0))/1000:.1f} bln PLN")
print(f"\nBucket totals (REMAINING - bln PLN):")
for b in pivot_12.columns:
    print(f"  {b}: {pivot_12[b].sum()/1000:.1f}")

In [ ]:
BUCKET_COLORS_12 = {
    "OS":    "#1f77b4",
    "Z":     "#ff7f0e",
    "I":     "#9467bd",
    "tbill": "#7f7f7f",
}

fig = go.Figure()

current_year = TODAY.year
for bucket in BUCKET_ORDER_12:
    if bucket in matured_by_bucket.index and matured_by_bucket[bucket] > 0:
        color = BUCKET_COLORS_12.get(bucket, "grey")
        fig.add_trace(go.Bar(
            x=[current_year], y=[matured_by_bucket[bucket]],
            name=f"{bucket} matured YTD",
            marker=dict(color=color, pattern=dict(shape="/", fgcolor="white", size=4)),
            opacity=0.55, showlegend=True,
            legendgroup=f"matured_{bucket}",
            hovertemplate=(f"<b>{bucket} MATURED YTD {current_year}</b><br>"
                           "%{y:,.0f} mln PLN<extra></extra>"),
        ))

for bucket in pivot_12.columns:
    fig.add_trace(go.Bar(
        x=pivot_12.index, y=pivot_12[bucket],
        name=bucket,
        marker_color=BUCKET_COLORS_12.get(bucket, "grey"),
        hovertemplate=(f"<b>{bucket}</b><br>%{{x}}: %{{y:,.0f}} mln PLN<extra></extra>"),
    ))

for year in sorted(set(pivot_12.index).union({current_year} if matured_ytd_total > 0 else set())):
    remaining = yearly_totals.get(year, 0)
    if year == current_year and matured_ytd_total > 0:
        total_for_year = remaining + matured_ytd_total
        label = f"<b>{total_for_year/1000:.1f}</b><br><span style='color:#666;font-size:9px'>(rem {remaining/1000:.1f} + mat {matured_ytd_total/1000:.1f})</span>"
    else:
        total_for_year = remaining
        label = f"<b>{total_for_year/1000:.1f}</b>"
    if total_for_year <= 0:
        continue
    fig.add_annotation(
        x=year, y=total_for_year, text=label, showarrow=False, yshift=14,
        font=dict(size=10, color="black"),
    )

fig.add_annotation(
    x=peak_year, y=0, text=f"PEAK<br>{peak_year}",
    showarrow=False, yshift=-20,
    font=dict(size=10, color="red", family="Arial Black"),
)

callout = (
    f"<b>Total outstanding: {total_outstanding/1000:.1f} bln PLN</b><br>"
    f"WAM: {wam_yrs:.2f} lat<br>"
    f"Next 12M: {mat_12m/1000:.1f} bln "
    f"({100*mat_12m/total_outstanding:.1f}% portfela)<br>"
    f"Matured YTD {current_year}: {matured_ytd_total/1000:.1f} bln"
)
fig.add_annotation(
    xref="paper", yref="paper", x=0.02, y=0.98, xanchor="left", yanchor="top",
    text=callout, showarrow=False, align="left",
    bgcolor="white", bordercolor="black", borderwidth=1, font=dict(size=11),
)

fig.update_layout(
    title=(f"Maturity ladder polskiego długu skarbowego (snapshot {TODAY.date()}) — "
           "solid = remaining, hatched = matured YTD"),
    xaxis_title="Rok wykupu", yaxis_title="Outstanding (mln PLN)",
    barmode="stack", template="plotly_white", height=600,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    xaxis=dict(dtick=1),
)
fig.show()

# LLM commentary
_top3_yrs = yearly_totals.sort_values(ascending=False).head(3)
_summary_12 = (
    f"Snapshot {TODAY.date()}:\n"
    f"  Total outstanding: {total_outstanding/1000:.1f} bln PLN\n"
    f"  WAM: {wam_yrs:.2f} lat\n"
    f"  Peak refinancing year: {peak_year} ({peak_val/1000:.1f} bln)\n"
    f"  Next 12M: {mat_12m/1000:.1f} bln ({100*mat_12m/total_outstanding:.1f}% portfela)\n"
    f"  Matured YTD {current_year}: {matured_ytd_total/1000:.1f} bln (zrolowane)\n\n"
    f"Top 3 peak years: " +
    ", ".join(f"{y}={v/1000:.0f}bln" for y, v in _top3_yrs.items())
)
display(Markdown(llm_chart_commentary(
    "Chart 5 — Maturity ladder (Bloomberg DDIS style)",
    "Stacked bar per rok wykupu, kolory per coupon bucket (OS/Z/I/tbill). Pokazuje 'wall of refinancing' - gdzie MF ma skoncentrowane wykupy. Hatched portion na current year = co juz zrolowane YTD. WAM (weighted avg maturity) i next-12M w callout.",
    _summary_12,
)))

## 6. Aukcje — statystyki per dzień (cały dzień)

Każda aukcja = wszystkie sprzedane tego dnia serie sumarycznie. Górny wykres: B/C, dolny: concession (cut-off vs same-day fixing 1).

In [ ]:
df4a = fetch_view(
    "v_auction_day_totals",
    f"?type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=*&order=auction_date.asc",
)
df4a["auction_date"] = pd.to_datetime(df4a["auction_date"])
for c in ["bid_to_cover", "bid_to_offer", "w_yield_avg",
          "w_concession_bp", "total_sold_mln", "total_demand_mln",
          "nc_share_demand"]:
    if c in df4a.columns:
        df4a[c] = pd.to_numeric(df4a[c], errors="coerce")

agg = df4a.groupby("auction_date", as_index=False).agg(
    total_sold_mln=("total_sold_mln", "sum"),
    total_demand_mln=("total_demand_mln", "sum"),
    w_yield_avg=("w_yield_avg", "mean"),
    w_concession_bp=("w_concession_bp", "mean"),
)
agg["bid_to_cover"] = agg["total_demand_mln"] / agg["total_sold_mln"]

# Rolling mean B/C - 12 aukcji ≈ kwartal (przy ~1 aukcji/tydzien). Window
# wyrownuje pojedyncze outliers (np. illiquid AZ z low demand), pokazuje
# strukturalny trend popytu vs jednoaukcyjne wahania.
MA_WINDOW = 12
agg = agg.sort_values("auction_date").reset_index(drop=True)
agg["bid_to_cover_ma"] = (
    agg["bid_to_cover"].rolling(window=MA_WINDOW, min_periods=4).mean()
)

print(f"Auction days: {len(agg)},  range: {agg.auction_date.min().date()} → {agg.auction_date.max().date()}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover + total sold", "Concession (bp)"),
                    specs=[[{"secondary_y": True}], [{}]])

fig.add_trace(go.Bar(x=agg["auction_date"], y=agg["total_sold_mln"],
                     name="Total sold (mln PLN)", marker_color="lightgrey",
                     opacity=0.6), row=1, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover"],
                         name="Bid-to-cover", mode="lines+markers",
                         line=dict(color="navy", width=1.5),
                         marker=dict(size=4),
                         opacity=0.55),
              row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover_ma"],
                         name=f"B/C MA({MA_WINDOW} aukcji)",
                         mode="lines",
                         line=dict(color="darkred", width=2.5)),
              row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["w_concession_bp"],
                         name="Concession (bp)", mode="lines+markers",
                         line=dict(color="seagreen", width=1.5)), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

fig.update_yaxes(title_text="B/C", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="mln PLN", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="bp", row=2, col=1)

fig.update_layout(
    title="Statystyki polskich aukcji obligacji skarbowych (AS+AU+AZ-sale)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.show()

# LLM commentary
_latest_4a = agg.iloc[-1]
_recent_4a = agg.tail(26)  # ~6m
_summary_4a = (
    f"Latest auction ({_latest_4a.auction_date.date()}):\n"
    f"  total sold {_latest_4a.total_sold_mln:,.0f} mln, "
    f"B/C {_latest_4a.bid_to_cover:.2f}, "
    f"concession {_latest_4a.w_concession_bp:+.1f} bp\n"
    f"MA({MA_WINDOW}) B/C: {_latest_4a.bid_to_cover_ma:.2f}\n\n"
    f"Last 6m (n={len(_recent_4a)}): "
    f"B/C mean {_recent_4a.bid_to_cover.mean():.2f} ±{_recent_4a.bid_to_cover.std():.2f}, "
    f"concession mean {_recent_4a.w_concession_bp.mean():+.1f}bp"
)
display(Markdown(llm_chart_commentary(
    "Chart 6 — Bid/cover + concession w czasie",
    "Time series wszystkich aukcji POLGB (AS+AU+AZ-sale): bid-to-cover (z MA12 jako structural demand trend) + concession w bp. Pokazuje ogólną siłę/słabość popytu na polski dług w czasie.",
    _summary_4a,
)))

## 7. B/C i concession per typ kuponu (I / OS / Z)

- **I** — inflation-linked (Oprocentowanie='I', np. IZ)
- **OS** — zero-coupon + stałe łącznie (O+S, np. OK + PS + DS + WS)
- **Z** — zmienne (WZ, NZ)

In [ ]:
df4b = fetch_view(
    "v_auction_by_coupon_bucket",
    f"?type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=*&order=auction_date.asc,coupon_bucket.asc",
)
df4b["auction_date"] = pd.to_datetime(df4b["auction_date"])
for c in ["bid_to_cover", "w_yield_avg", "w_concession_bp", "total_sold_mln"]:
    df4b[c] = pd.to_numeric(df4b[c], errors="coerce")

print(f"Rows: {len(df4b)},  buckets: {sorted(df4b.coupon_bucket.unique())}")

BUCKET_COLORS = {"I": "darkviolet", "OS": "navy", "Z": "darkorange"}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover", "Concession (bp)"))

for bucket in sorted(df4b.coupon_bucket.unique()):
    sub = df4b[df4b.coupon_bucket == bucket].sort_values("auction_date")
    color = BUCKET_COLORS.get(bucket, "grey")
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["bid_to_cover"],
                             name=bucket, legendgroup=bucket, mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["w_concession_bp"],
                             name=bucket, legendgroup=bucket, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)
fig.update_layout(
    title="Metryki aukcyjne per kubełek kuponu (AS+AU+AZ-sale)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.show()

# LLM commentary
_4b_latest = df4b[df4b.auction_date == df4b.auction_date.max()]
_4b_lines = []
for _, r in _4b_latest.iterrows():
    _4b_lines.append(
        f"  {r.coupon_bucket}: B/C {r.bid_to_cover:.2f}, "
        f"conc {r.w_concession_bp:+.1f}bp, sold {r.total_sold_mln:,.0f} mln"
    )
_summary_4b = (
    f"Latest auction date {df4b.auction_date.max().date()}:\n" +
    "\n".join(_4b_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 7 — B/C + concession per coupon bucket",
    "Time series B/C i concession rozbite per coupon bucket (I/OS/Z). Pozwala porownac sile popytu na rozne typy papieru — np. czy floatery (Z) maja stabilnie wyzsze concession niz fixed (OS).",
    _summary_4b,
)))

## 8. Aukcje — popyt, podaż, wykonanie

Trzy stacked subplots (shared X), analog skryptu R:

- **6a** (60% wysokości) — **AS primary**: candlestick zakresu `PodażMin → PodażMax` jako rectangle (bez wicków), zielony `+` = Popyt, czerwony `×` = Sprzedano, czarne dashed connector między nimi, plus rolling mean (window=6) dla obu.
- **6b** (20%) — **AU top-up** (= R `SD`, sprzedaż dodatkowa): bar = `PodażMax`, czerwone `×` = sprzedaż, rolling mean.
- **6c** (20%) — **NK z AS** (non-competitive): zielony `+` Popyt NK, czerwony `×` Sprzedaż NK, rolling means dla obu.

Pozwala zobaczyć jednym rzutem oka: czy MF cieli targety (top vs bottom z range), czy popyt rośnie/spada strukturalnie, jak duża frakcja idzie do non-competitive bidders.

In [ ]:
# Raw bond_auctions sale legs - potrzebujemy offer_min/max ktorych v_auction_day_totals
# nie agreguje (tylko offer_max). Agregacja per (data, type_op) w Python.
#
# UWAGA: MF zmienilo nomenklature top-upu ok. 2018: stara nazwa AU, nowa SD
# (Sprzedaz Dodatkowa). Identyczna semantyka, traktujemy obie jako "top-up"
# i zlewamy w jeden bucket przed agregacja.
raw6 = fetch_view(
    "bond_auctions",
    f"?type_tx=eq.S&type_op=in.(AS,AU,SD)&auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,type_op,offer_min_mln,offer_max_mln,"
    "demand_total_mln,demand_nc_mln,sold_total_mln,sold_nc_mln"
    "&order=auction_date.asc",
)
raw6["auction_date"] = pd.to_datetime(raw6["auction_date"])
for c in ["offer_min_mln", "offer_max_mln", "demand_total_mln",
          "demand_nc_mln", "sold_total_mln", "sold_nc_mln"]:
    raw6[c] = pd.to_numeric(raw6[c], errors="coerce")

# Zlej SD → AU (ten sam top-up, inna nomenklatura)
raw6["type_op_norm"] = raw6["type_op"].replace({"SD": "AU"})

# Agregat per (date, normalized type_op) - sum across series tego samego dnia
agg6 = raw6.groupby(["auction_date", "type_op_norm"], as_index=False).agg(
    offer_min_mln=("offer_min_mln", "sum"),
    offer_max_mln=("offer_max_mln", "sum"),
    demand_mln=("demand_total_mln", "sum"),
    demand_nc_mln=("demand_nc_mln", "sum"),
    sold_mln=("sold_total_mln", "sum"),
    sold_nc_mln=("sold_nc_mln", "sum"),
).rename(columns={"type_op_norm": "type_op"})

ROLL_WIN_6 = 6
def _with_roll(df):
    df = df.sort_values("auction_date").reset_index(drop=True)
    for col in ["demand_mln", "sold_mln", "demand_nc_mln", "sold_nc_mln"]:
        df[f"roll_{col}"] = df[col].rolling(ROLL_WIN_6, min_periods=2).mean()
    return df

df6_as = _with_roll(agg6[agg6["type_op"] == "AS"].copy())
df6_au = _with_roll(agg6[agg6["type_op"] == "AU"].copy())

print(f"AS auctions: {len(df6_as)} "
      f"({df6_as['auction_date'].min().date()} → {df6_as['auction_date'].max().date()})")
print(f"AU+SD (top-up) auctions: {len(df6_au)}")
if len(df6_au):
    raw_au = raw6[raw6["type_op"] == "AU"]["auction_date"].nunique()
    raw_sd = raw6[raw6["type_op"] == "SD"]["auction_date"].nunique()
    n_pm = (df6_au["offer_max_mln"].fillna(0) > 0).sum()
    n_sold = (df6_au["sold_mln"].fillna(0) > 0).sum()
    print(f"  source breakdown: AU dates={raw_au}, SD dates={raw_sd}")
    print(f"  offer_max_mln populated: {n_pm}/{len(df6_au)}  "
          f"sold_mln populated: {n_sold}/{len(df6_au)}")
    print(f"  date range: {df6_au['auction_date'].min().date()} → "
          f"{df6_au['auction_date'].max().date()}")
df6_as.tail()

In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.2, 0.2],
    vertical_spacing=0.04,
    subplot_titles=(
        "Aukcja AS — zakres podaży (candle) + Popyt / Sprzedano + MA(6)",
        "Aukcja AU (top-up) — Sprzedaż + MA(6) [PodażMax markowane gdy w MF]",
        "Non-competitive (NK) z AS — Popyt + Sprzedaż + MA(6)",
    ),
)

# ============================== 6a: AS primary ==============================
fig.add_trace(go.Candlestick(
    x=df6_as["auction_date"],
    open=df6_as["offer_min_mln"], close=df6_as["offer_max_mln"],
    low=df6_as["offer_min_mln"], high=df6_as["offer_max_mln"],
    name="PodażMin/Max",
    increasing_line_color="lightsteelblue",
    decreasing_line_color="lightsteelblue",
    increasing_fillcolor="lightsteelblue",
    decreasing_fillcolor="lightsteelblue",
), row=1, col=1)

# Dashed connector kazdej aukcji: Sell → Demand
for d, s_, dem in zip(df6_as["auction_date"], df6_as["sold_mln"], df6_as["demand_mln"]):
    if pd.notna(s_) and pd.notna(dem):
        fig.add_shape(
            type="line",
            x0=d, x1=d, y0=s_, y1=dem,
            line=dict(color="black", width=1, dash="dash"),
            row=1, col=1,
        )

fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["sold_mln"],
    mode="markers", marker=dict(symbol="x", size=9, color="red"),
    name="Sprzedano",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["demand_mln"],
    mode="markers", marker=dict(symbol="cross", size=9, color="green"),
    name="Popyt",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["roll_demand_mln"],
    mode="lines", line=dict(color="green", width=2),
    name=f"Popyt — Średnia ({ROLL_WIN_6})",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["roll_sold_mln"],
    mode="lines", line=dict(color="red", width=2),
    name=f"Sprzedaż — Średnia ({ROLL_WIN_6})",
), row=1, col=1)

# ============================== 6b: AU top-up ==============================
BAR_WIDTH_MS = 86400000 * 4
fig.add_trace(go.Bar(
    x=df6_au["auction_date"], y=df6_au["sold_mln"],
    width=BAR_WIDTH_MS,
    marker_color="lightsteelblue",
    marker_line=dict(color="steelblue", width=0.5),
    name="Top-up Sprzedaż", showlegend=False,
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df6_au["auction_date"], y=df6_au["offer_max_mln"],
    mode="markers",
    marker=dict(symbol="triangle-down", size=8, color="black"),
    name="Top-up PodażMax (gdy w MF)", showlegend=False,
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df6_au["auction_date"], y=df6_au["roll_sold_mln"],
    mode="lines", line=dict(color="red", width=2),
    name=f"Top-up Sprzedaż — Średnia ({ROLL_WIN_6})", showlegend=False,
), row=2, col=1)

# ============================== 6c: NK z AS ==============================
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["sold_nc_mln"],
    mode="markers", marker=dict(symbol="x", size=9, color="red"),
    name="Sprzedano NK", showlegend=False,
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["demand_nc_mln"],
    mode="markers", marker=dict(symbol="cross", size=9, color="green"),
    name="Popyt NK", showlegend=False,
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["roll_sold_nc_mln"],
    mode="lines+markers", line=dict(color="red", width=2),
    marker=dict(symbol="circle", size=4, color="red"),
    name=f"Sprzedaż NK — Średnia ({ROLL_WIN_6})", showlegend=False,
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df6_as["auction_date"], y=df6_as["roll_demand_nc_mln"],
    mode="lines+markers", line=dict(color="green", width=2),
    marker=dict(symbol="circle", size=4, color="green"),
    name=f"Popyt NK — Średnia ({ROLL_WIN_6})", showlegend=False,
), row=3, col=1)

fig.update_layout(
    title="Aukcje obligacji skarbowych — popyt, podaż, wykonanie",
    template="plotly_white", height=950,
    hovermode="x unified",
    showlegend=True,
    legend=dict(orientation="h", y=-0.06),
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=False,
    xaxis3_rangeslider_visible=False,
)
fig.update_yaxes(title_text="mln PLN", row=1, col=1)
fig.update_yaxes(title_text="mln PLN", row=2, col=1)
fig.update_yaxes(title_text="mln PLN", row=3, col=1)
fig.show()

# LLM commentary
_6_latest_as = df6_as.iloc[-1] if len(df6_as) else None
_6_latest_au = df6_au.iloc[-1] if len(df6_au) else None
_6_lines = []
if _6_latest_as is not None:
    _6_lines.append(
        f"Latest AS ({_6_latest_as.auction_date.date()}): "
        f"offer {_6_latest_as.offer_min_mln:,.0f}-{_6_latest_as.offer_max_mln:,.0f} mln, "
        f"demand {_6_latest_as.demand_mln:,.0f}, sold {_6_latest_as.sold_mln:,.0f}, "
        f"NK demand {_6_latest_as.demand_nc_mln:,.0f}, NK sold {_6_latest_as.sold_nc_mln:,.0f}"
    )
    _6_lines.append(
        f"AS MA(6): demand {_6_latest_as.roll_demand_mln:,.0f}, "
        f"sold {_6_latest_as.roll_sold_mln:,.0f}"
    )
if _6_latest_au is not None:
    _6_lines.append(
        f"Latest AU ({_6_latest_au.auction_date.date()}): "
        f"sold {_6_latest_au.sold_mln:,.0f}, MA(6) sold {_6_latest_au.roll_sold_mln:,.0f}"
    )
_summary_6 = "\n".join(_6_lines) if _6_lines else "no recent auctions"
display(Markdown(llm_chart_commentary(
    "Chart 8 — Aukcje popyt/podaż/wykonanie",
    "3 panele: AS primary (candle podazy + popyt/sprzedaz markers + dashed connector + MA6), AU top-up (bar sprzedazy + MA6), NK z AS (popyt/sprzedaz non-competitive + MA6). Pokazuje czy MF cieli targety, jak duza frakcja jest non-comp, jak ewoluuja absolute volumes.",
    _summary_6,
)))

## 9. Ostatnia aukcja — szczegóły per seria

Tabela wszystkich serii sprzedanych w najnowszej dacie. `concession_bp` > 0 = aukcja droga vs rynek wtórny, < 0 = "through" (taniej niż rynek).

In [ ]:
df4d_recent = fetch_view(
    "v_recent_auctions",
    "?type_op=in.(AS,AU,AZ)&select=*&order=auction_date.desc&limit=50",
)
df4d_recent["auction_date"] = pd.to_datetime(df4d_recent["auction_date"]).dt.date

latest_date = df4d_recent["auction_date"].max()
df_last = df4d_recent[df4d_recent["auction_date"] == latest_date].copy()
for c in ["offer_max_mln", "demand_total_mln", "sold_total_mln",
          "bid_to_cover", "yield_avg",
          "concession_bp", "nc_share_demand"]:
    if c in df_last.columns:
        df_last[c] = pd.to_numeric(df_last[c], errors="coerce")

total_sold = df_last["sold_total_mln"].sum()
total_demand = df_last["demand_total_mln"].sum()
overall_bc = total_demand / total_sold if total_sold else float("nan")

print(f"Ostatnia aukcja: {latest_date}  ({len(df_last)} serii)")
print(f"Łączny sold:    {total_sold:>10,.1f} mln PLN")
print(f"Łączny demand:  {total_demand:>10,.1f} mln PLN")
print(f"B/C całej aukcji: {overall_bc:.2f}")

cols = ["seria", "type_op", "years_to_maturity", "coupon_kind", "offer_max_mln",
        "demand_total_mln", "sold_total_mln", "bid_to_cover",
        "yield_avg", "concession_bp", "nc_share_demand"]
view = df_last[cols].rename(columns={
    "type_op": "op",
    "years_to_maturity": "yrs",
    "coupon_kind": "kind",
    "offer_max_mln": "offer",
    "demand_total_mln": "demand",
    "sold_total_mln": "sold",
    "bid_to_cover": "B/C",
    "yield_avg": "yld_avg %",
    "concession_bp": "conc bp",
    "nc_share_demand": "NK %",
})
view["NK %"] = view["NK %"] * 100
display(view.style.format({
    "offer": "{:,.0f}", "demand": "{:,.0f}", "sold": "{:,.0f}",
    "B/C": "{:.2f}", "yld_avg %": "{:.3f}",
    "conc bp": "{:+.1f}", "NK %": "{:.1f}",
}, na_rep="—"))

# LLM commentary
_4c_lines = []
for _, r in df_last.iterrows():
    _4c_lines.append(
        f"  {r.seria} ({r.type_op}, {r.years_to_maturity}Y, {r.coupon_kind}): "
        f"sold {r.sold_total_mln:,.0f}, B/C {r.bid_to_cover:.2f}, "
        f"yld {r.yield_avg:.3f}%, conc {r.concession_bp:+.1f}bp"
    )
_summary_4c = (
    f"Ostatnia aukcja {latest_date} - {len(df_last)} serii:\n"
    f"Total sold: {total_sold:,.0f} mln, demand: {total_demand:,.0f} mln, "
    f"overall B/C: {overall_bc:.2f}\n\nPer seria:\n" + "\n".join(_4c_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 9 — Tabela ostatniej aukcji",
    "Per-seria szczegoly z ostatniej aukcji: offer, demand, sold, B/C, yield_avg, concession w bp, NK share. Pokazuje break-down 'co sie stalo' na konkretnych ISIN-ach.",
    _summary_4c,
)))

## 10. Historyczna dystrybucja per seria per seria — box ploty

Dla każdej serii sprzedanej w ostatniej aukcji (4c):
- **Box plot** całej historii (od 2012) tej **konkretnej** serii: wszystkie wcześniejsze aukcje (AS+AU+AZ-sale).
- **Czerwony diament** = wartość z dzisiejszej aukcji — od razu widać czy dzisiaj jest "typowo", ekstremalnie czy outlier.

Pusty box (sama czerwona kropka) = brand new series, brak historii.

Dwie metryki:
- góra: **Bid-to-cover** (referencyjna linia y=1)
- dół: **Concession bp** (referencyjna linia y=0)

In [ ]:
# Wszystkie historyczne sale legs (AS+AU+AZ) od 2012-01-01 z B/C i concession.
# v_recent_auctions juz filtruje type_tx='S' AND type_op IN ('AS','AU','AZ').
all_hist = fetch_view(
    "v_recent_auctions",
    f"?auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,seria,bid_to_cover,concession_bp"
    "&order=auction_date.asc",
)
all_hist["auction_date"] = pd.to_datetime(all_hist["auction_date"]).dt.date
all_hist["bid_to_cover"] = pd.to_numeric(all_hist["bid_to_cover"], errors="coerce")
all_hist["concession_bp"] = pd.to_numeric(all_hist["concession_bp"], errors="coerce")

# Lista serii sprzedanych dzisiaj (z df_last w chart 4c, juz w scope)
today_series_list = sorted(df_last["seria"].dropna().unique().tolist())

# Filter do tych serii, wyklucz dzisiejsza aukcje (zeby box nie zawieral
# punktu ktory rysujemy osobno czerwonym diamentem)
df_hist = all_hist[
    all_hist["seria"].isin(today_series_list)
    & (all_hist["auction_date"] < latest_date)
].copy()

print(f"Today's series ({latest_date}): {today_series_list}")
print(f"Total historical auctions of these series: {len(df_hist)}")
for s in today_series_list:
    n = len(df_hist[df_hist["seria"] == s])
    print(f"  {s}: {n} prior auctions")

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=("Historyczne Bid-to-cover per seria (od 2012)",
                    "Historyczne Concession bp per seria (od 2012)"),
)

today_marker = dict(
    symbol="diamond", size=14, color="red",
    line=dict(color="black", width=1),
)

for i, s in enumerate(today_series_list):
    hist = df_hist[df_hist.seria == s]
    today_row = df_last[df_last.seria == s].iloc[0]
    today_bc = pd.to_numeric(today_row.get("bid_to_cover"), errors="coerce")
    today_conc = pd.to_numeric(today_row.get("concession_bp"), errors="coerce")
    n_hist = len(hist)

    # Box B/C
    fig.add_trace(go.Box(
        y=hist["bid_to_cover"], name=s,
        marker_color="lightblue", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:.2f}<extra></extra>"
        ),
    ), row=1, col=1)
    # Today's value B/C marker
    if pd.notna(today_bc):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_bc],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=(i == 0),
            hovertemplate=f"<b>{s}</b><br>dzisiaj B/C: %{{y:.2f}}<extra></extra>",
        ), row=1, col=1)

    # Box concession
    fig.add_trace(go.Box(
        y=hist["concession_bp"], name=s,
        marker_color="lightgreen", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:+.1f} bp<extra></extra>"
        ),
    ), row=2, col=1)
    # Today's value concession marker
    if pd.notna(today_conc):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_conc],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=False,
            hovertemplate=f"<b>{s}</b><br>dzisiaj conc: %{{y:+.1f}} bp<extra></extra>",
        ), row=2, col=1)

fig.add_hline(y=1, line_dash="dot", line_color="grey", row=1, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="grey", row=2, col=1)

fig.update_layout(
    title=f"Aukcja {latest_date}: dzisiaj vs pełna historia per seria",
    template="plotly_white",
    height=750,
    showlegend=True,
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.update_xaxes(title_text="Seria", row=2, col=1)
fig.show()

# LLM commentary
_4d_lines = []
for s in today_series_list:
    hist = df_hist[df_hist.seria == s]
    today_row = df_last[df_last.seria == s].iloc[0]
    bc_today = pd.to_numeric(today_row.get("bid_to_cover"), errors="coerce")
    conc_today = pd.to_numeric(today_row.get("concession_bp"), errors="coerce")
    bc_med = hist["bid_to_cover"].median() if len(hist) else float("nan")
    conc_med = hist["concession_bp"].median() if len(hist) else float("nan")
    _4d_lines.append(
        f"  {s} (n={len(hist)}): "
        f"B/C today {bc_today:.2f} vs median {bc_med:.2f}, "
        f"conc today {conc_today:+.1f}bp vs median {conc_med:+.1f}bp"
    )
_summary_4d = (
    f"Aukcja {latest_date} - serie vs ich pelna historia (od 2012):\n" +
    "\n".join(_4d_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 10 — Box ploty per seria z dzisiejsza aukcja",
    "Per kazda seria dzisiaj sprzedana: box plot wszystkich poprzednich aukcji tej serii (B/C i concession) + czerwony diament dzisiaj. Pokazuje czy print byl typowy czy outlier w kontekscie historycznym tej konkretnej serii.",
    _summary_4d,
)))

## 11. Wpływ aukcji na portfolio metryki

Każda aukcja sprzedażowa (AS/AU) zmienia ważone outstanding metryki portfela: ATM, ATR, Mod/Mac Duration. Wykres pokazuje **pure composition impact** — delta metryki na dzień aukcji vs poprzedni dzień handlowy z odjętą time-decay (~1/365 per dzień kalendarzowy).

Interpretacja:
- **Δ > 0** (niebieski) — aukcja **wydłużyła** średnią charakterystykę portfela (typowo nowa długa seria DS/WS/IZ)
- **Δ < 0** (czerwony) — aukcja **skróciła** (np. OK krótka, albo re-open krótkiej istniejącej serii)
- **Δ ≈ 0** — re-open serii z metryką bliską portfolio average (nominal change tylko wagi)

ATM ma czysty time decay (wszyscy = -1/365 dziennie) więc po korekcie zostaje praktycznie wyłącznie composition impact. Dla Mod/Mac jest to przybliżenie (zakłada flat yield curve day-over-day).

In [ ]:
# Fetch aukcje (sale legs AS/AU/AZ-sale) z listami serii per dzien.
auctions_5 = fetch_view(
    "bond_auctions",
    f"?type_tx=eq.S&type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,seria,sold_total_mln&order=auction_date.asc",
)
auctions_5["auction_date"] = pd.to_datetime(auctions_5["auction_date"])
auctions_5["sold_total_mln"] = pd.to_numeric(auctions_5["sold_total_mln"], errors="coerce")

day_agg_5 = auctions_5.groupby("auction_date").agg(
    total_sold_mln=("sold_total_mln", "sum"),
    series_list=("seria", lambda s: ", ".join(sorted(set(s)))),
).reset_index().set_index("auction_date")

# df1 z chart 1 (portfolio_metrics_daily) - liczymy delta per metryka
# vs poprzedni fixing day + day gap dla time-decay correction.
mts = df1.set_index("fixing_date").sort_index()
PORT_COLS = ["portfolio_mod_duration", "portfolio_mac_duration",
             "portfolio_atm", "portfolio_atr"]
for col in PORT_COLS:
    mts[f"delta_{col}"] = mts[col].diff()
mts["gap_days"] = pd.Series(mts.index, index=mts.index).diff().dt.days

# Join na auction_date. Auction zwykle jest tez fixing day (BondSpot codziennie
# pon-pt) wiec to powinno match-owac dla 99% aukcji.
df5 = day_agg_5.join(mts[[f"delta_{c}" for c in PORT_COLS] + ["gap_days"]], how="inner")

# Pure composition impact = raw delta + time-decay correction (gap_days/365).
# Dla ATM to identycznosc matematyczna (wszyscy bondy traca 1/calendar_year),
# dla Mod/Mac przyblizenie (zakladamy flat YC d-o-d).
for col in PORT_COLS:
    df5[f"impact_{col}"] = df5[f"delta_{col}"] + df5["gap_days"] / 365.0

print(f"Auctions joined with portfolio metrics: {len(df5)} "
      f"(range {df5.index.min().date()} → {df5.index.max().date()})")
print(f"Avg impact: mod={df5['impact_portfolio_mod_duration'].mean():+.4f}Y, "
      f"atm={df5['impact_portfolio_atm'].mean():+.4f}Y")
print(f"Biggest single impact (mod_duration): "
      f"{df5['impact_portfolio_mod_duration'].abs().idxmax().date()} "
      f"({df5['impact_portfolio_mod_duration'].abs().max():.4f}Y)")
df5.tail(10)

In [ ]:
metrics_5 = [
    ("impact_portfolio_mod_duration", "Δ Modified Duration (lata)"),
    ("impact_portfolio_mac_duration", "Δ Macaulay Duration (lata)"),
    ("impact_portfolio_atm",          "Δ ATM (lata)"),
    ("impact_portfolio_atr",          "Δ ATR (lata)"),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics_5],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

# Niebieski dla +, czerwony dla -, szary dla blisko zera (|Δ| < 1bp ATM)
def _color(v):
    if pd.isna(v):
        return "lightgrey"
    if abs(v) < 0.0001:
        return "lightgrey"
    return "#1f77b4" if v > 0 else "#d62728"

customdata = list(zip(df5["total_sold_mln"], df5["series_list"]))
df5_sorted = df5.sort_index()  # do rolling
ROLL_Q_5 = 13  # ~kwartal przy 1 aukcji/tydzien

for i, (col, label) in enumerate(metrics_5):
    row, c = i // 2 + 1, i % 2 + 1
    colors = [_color(v) for v in df5[col]]
    fig.add_trace(
        go.Bar(
            x=df5.index, y=df5[col],
            marker_color=colors,
            name=label, showlegend=False,
            customdata=customdata,
            hovertemplate=(
                "<b>%{x|%Y-%m-%d}</b><br>"
                f"{label}: " + "%{y:+.4f} Y<br>"
                "sold: %{customdata[0]:,.0f} mln PLN<br>"
                "series: %{customdata[1]}<extra></extra>"
            ),
        ),
        row=row, col=c,
    )

    # Rolling mean ~kwartal (13 aukcji przy ~1 aukcji/tydzien). Smooth line
    # over bars - latwo wylapac structural quarterly tilt vs single-auction noise.
    q_roll = df5_sorted[col].rolling(ROLL_Q_5, min_periods=4).mean()
    fig.add_trace(go.Scatter(
        x=q_roll.index, y=q_roll.values,
        mode="lines",
        line=dict(color="black", width=2.5),
        name=f"Rolling mean ({ROLL_Q_5} aukcji ≈ kwartał)",
        legendgroup="q_roll",
        showlegend=(i == 0),
        hovertemplate="<b>%{x|%Y-%m-%d}</b><br>roll mean: %{y:+.4f} Y<extra></extra>",
    ), row=row, col=c)

    fig.add_hline(y=0, line_dash="dot", line_color="grey", row=row, col=c)

fig.update_layout(
    title=(f"Wpływ poszczególnych aukcji na portfolio metryki "
           f"(composition impact, time-decay odjete) + rolling mean ({ROLL_Q_5} aukcji)"),
    template="plotly_white",
    height=750,
    hovermode="closest",
    bargap=0.1,
    legend=dict(orientation="h", y=-0.08),
)
fig.show()

# LLM commentary
_5_latest = df5.iloc[-1]
_5_recent = df5_sorted.tail(ROLL_Q_5)
_summary_5 = (
    f"Latest auction ({_5_latest.name.date()}) impact na portfolio:\n"
    f"  Δ Mod Duration: {_5_latest.impact_portfolio_mod_duration:+.4f}Y\n"
    f"  Δ Mac Duration: {_5_latest.impact_portfolio_mac_duration:+.4f}Y\n"
    f"  Δ ATM:          {_5_latest.impact_portfolio_atm:+.4f}Y\n"
    f"  Δ ATR:          {_5_latest.impact_portfolio_atr:+.4f}Y\n"
    f"  Sold: {_5_latest.total_sold_mln:,.0f} mln; series: {_5_latest.series_list}\n\n"
    f"Rolling mean ostatnie {ROLL_Q_5} aukcji (~kwartal):\n"
    f"  Δ ATM: {_5_recent['impact_portfolio_atm'].mean():+.4f}Y\n"
    f"  Δ Mod: {_5_recent['impact_portfolio_mod_duration'].mean():+.4f}Y"
)
display(Markdown(llm_chart_commentary(
    "Chart 11 — Wplyw aukcji na portfolio metryki",
    "Per-aukcja bary pokazuja jak konkretna aukcja zmienila weighted portfolio Mod Duration, Mac Duration, ATM, ATR (composition impact, z odjeta time-decay 1/365). Rolling mean linia pokazuje strukturalny quarterly tilt - czy MF systematycznie wydluza/skraca dlug.",
    _summary_5,
)))

## 12. Tail metric — concession per aukcja z kolorowaniem

Tail / concession = ile auction wyceniono drożej (czerwony) lub taniej (zielony, "through") niż BondSpot przed aukcją. Standardowa miara siły na UST/Bund desks.

Bar chart per aukcja + rolling 6-auction mean + ±1 SD band — outliers spoza pasma to weak/strong prints.

- **Bar zielony** (concession < 0): "stops through" — aukcja niżej niż rynek = silny popyt
- **Bar czerwony** (concession > 0): "tail" — MF musiało podnieść yield, słabszy popyt
- **Czarna linia** + szary band: średnia 6 aukcji ±1 SD — typowy zakres tego okresu

In [ ]:
# Per-aukcja concession (z df4a juz pogrupowanego per (data, type_op)).
# Daily aggregate = mean across type_op tego dnia (zwykle 1 type_op/dzien).
df7 = df4a[["auction_date", "w_concession_bp", "total_sold_mln"]].dropna(
    subset=["w_concession_bp"]
).copy()
day_conc = df7.groupby("auction_date", as_index=False).agg(
    concession_bp=("w_concession_bp", "mean"),
    sold_mln=("total_sold_mln", "sum"),
).sort_values("auction_date").reset_index(drop=True)

WIN_7 = 6
day_conc["roll_mean"] = day_conc["concession_bp"].rolling(WIN_7, min_periods=3).mean()
day_conc["roll_std"] = day_conc["concession_bp"].rolling(WIN_7, min_periods=3).std()

print(f"Auctions z concession: {len(day_conc)}, "
      f"range {day_conc.auction_date.min().date()} → {day_conc.auction_date.max().date()}")
print(f"Latest concession: {day_conc.iloc[-1]['concession_bp']:+.1f} bp "
      f"(roll mean {day_conc.iloc[-1]['roll_mean']:+.1f} ±{day_conc.iloc[-1]['roll_std']:.1f})")

fig = go.Figure()

# ±1 SD band (rysujemy najpierw zeby bary szly na wierzchu)
fig.add_trace(go.Scatter(
    x=day_conc["auction_date"], y=day_conc["roll_mean"] + day_conc["roll_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    showlegend=False, hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=day_conc["auction_date"], y=day_conc["roll_mean"] - day_conc["roll_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(120,120,120,0.18)",
    name=f"±1 SD ({WIN_7}-auction)",
    hoverinfo="skip",
))

# Bary kolorowane
colors_7 = ["#2ca02c" if v < 0 else "#d62728" for v in day_conc["concession_bp"]]
fig.add_trace(go.Bar(
    x=day_conc["auction_date"], y=day_conc["concession_bp"],
    marker_color=colors_7,
    name="Concession per aukcja",
    hovertemplate=(
        "%{x|%Y-%m-%d}<br>"
        "concession: %{y:+.1f} bp<br>"
        "sold: %{customdata:,.0f} mln<extra></extra>"
    ),
    customdata=day_conc["sold_mln"],
))

# Rolling mean line
fig.add_trace(go.Scatter(
    x=day_conc["auction_date"], y=day_conc["roll_mean"],
    mode="lines", line=dict(color="black", width=2),
    name=f"Średnia {WIN_7}-aukcji",
))

fig.add_hline(y=0, line_dash="dash", line_color="grey")

fig.update_layout(
    title="Tail / concession per aukcja — zielony = through (silny popyt), czerwony = tail (słaby)",
    xaxis_title="Data aukcji",
    yaxis_title="Concession (bp)",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_7_latest = day_conc.iloc[-1]
_7_recent = day_conc.tail(WIN_7 * 2)
_through_n = (_7_recent["concession_bp"] < 0).sum()
_tail_n = (_7_recent["concession_bp"] > 0).sum()
_summary_7 = (
    f"Latest auction ({_7_latest.auction_date.date()}): "
    f"concession {_7_latest.concession_bp:+.1f} bp; "
    f"MA({WIN_7}) mean {_7_latest.roll_mean:+.1f} ±{_7_latest.roll_std:.1f}\n"
    f"Z-score dzisiejszego print: "
    f"{(_7_latest.concession_bp - _7_latest.roll_mean) / _7_latest.roll_std:+.2f}σ\n\n"
    f"Last {len(_7_recent)} aukcji: {_through_n} through (zielony), {_tail_n} tail (czerwony)"
)
display(Markdown(llm_chart_commentary(
    "Chart 12 — Tail / concession per aukcja",
    "Bar chart per aukcja: zielone = stops through (silny popyt, conc<0), czerwone = tail (slabszy, conc>0). Linia MA(6) + szare pasmo ±1SD pokazuje typowy zakres - outliery sygnalizuja exceptional prints.",
    _summary_7,
)))

## 13. Auction scorecard — dzisiejsza aukcja vs rolling 6m / 12m

Per coupon bucket (I/OS/Z), porównanie metryk dzisiejszej aukcji vs rolling means±std obliczone z 26 i 52 wcześniejszych aukcji tego bucketu (~6m i ~12m przy ~1 aukcji/tydzień).

**Kolumna `kierunek`** — pokazuje którą stronę "lepiej" interpretuje się dla danej metryki:
- **B/C** `↑ better` — wyższy = silniejszy popyt
- **Concession bp** `↓ better` — niższy/ujemny = aukcja "through" (silny popyt)
- **NK share %** `↑ better` — więcej genuine end-user demand

**Kolor komórki `z`** — kolorowany po **favorable direction** (świadomy znaku per metryka):
- 🟢 zielony = **lepiej niż typowo** (favorable_z ≥ +1)
- 🔴 czerwony = **gorzej niż typowo** (favorable_z ≤ -1)
- biały = w paśmie ±1σ

Ciemniejszy odcień gdy |favorable_z| ≥ 2 (ekstremum).

In [ ]:
# Trzeba popytu NK na poziomie bucket - df4b ma juz B/C i concession per bucket,
# ale nie ma NK share. Doliczymy w Python z bond_auctions raw.
sc_raw = fetch_view(
    "bond_auctions",
    f"?type_tx=eq.S&type_op=in.(AS,AU,SD,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,isin,coupon_kind,sold_total_mln,demand_total_mln,"
    "demand_nc_mln&order=auction_date.asc",
)
sc_raw["auction_date"] = pd.to_datetime(sc_raw["auction_date"])
for c in ["sold_total_mln", "demand_total_mln", "demand_nc_mln"]:
    sc_raw[c] = pd.to_numeric(sc_raw[c], errors="coerce")
sc_raw["sold_abs"] = sc_raw["sold_total_mln"].abs()

def _bucket(k):
    if k == "I": return "I"
    if k == "Z": return "Z"
    if k in ("S", "O"): return "OS"
    return None
sc_raw["bucket"] = sc_raw["coupon_kind"].map(_bucket)

# Per (date, bucket): demand, sold, nk_share
sc_agg = sc_raw.dropna(subset=["bucket"]).groupby(
    ["auction_date", "bucket"], as_index=False
).agg(
    sold=("sold_abs", "sum"),
    demand=("demand_total_mln", "sum"),
    nk_demand=("demand_nc_mln", "sum"),
)
sc_agg["bid_to_cover"] = sc_agg["demand"] / sc_agg["sold"].replace(0, pd.NA)
sc_agg["nk_share_pct"] = sc_agg["nk_demand"] / sc_agg["demand"].replace(0, pd.NA) * 100

# Concession z df4b (per bucket; coupon_bucket label match)
conc_per_bucket = df4b[["auction_date", "coupon_bucket", "w_concession_bp"]].rename(
    columns={"coupon_bucket": "bucket", "w_concession_bp": "concession_bp"}
)
sc_agg = sc_agg.merge(conc_per_bucket, on=["auction_date", "bucket"], how="left")

print(f"sc_agg rows: {len(sc_agg)}, "
      f"buckets: {sorted(sc_agg['bucket'].dropna().unique()) if len(sc_agg) else '(empty)'}")

if sc_agg.empty:
    print("! sc_agg jest pusty - brak aukcji w bond_auctions po filtrze. "
          "Sprawdz czy bond_auctions ma rowy AS/AU/SD/AZ od START_DATE.")
else:
    # Compute z-scores dla ostatniej aukcji per bucket
    latest_date_sc = sc_agg["auction_date"].max()
    WIN_6M, WIN_12M = 26, 52
    # (column, label, sign +1=higher better, -1=lower better, kierunek_hint)
    metric_specs = [
        ("bid_to_cover",  "B/C",           +1, "↑ better"),
        ("concession_bp", "Concession bp", -1, "↓ better"),
        ("nk_share_pct",  "NK share %",    +1, "↑ better"),
    ]
    metric_sign = {label: sign for _, label, sign, _ in metric_specs}
    metric_hint = {label: hint for _, label, _, hint in metric_specs}

    records_8 = []
    for bucket in sorted(sc_agg["bucket"].dropna().unique()):
        sub = sc_agg[sc_agg.bucket == bucket].sort_values("auction_date")
        today_rows = sub[sub.auction_date == latest_date_sc]
        if today_rows.empty:
            continue
        today_row = today_rows.iloc[0]
        hist_all = sub[sub.auction_date < latest_date_sc]
        for col, label, sign, _ in metric_specs:
            today_val = today_row[col]
            if pd.isna(today_val):
                continue
            for win_label, win in [("6m", WIN_6M), ("12m", WIN_12M)]:
                hist = hist_all.tail(win)[col].dropna()
                if len(hist) < 3:
                    continue
                mean, std = hist.mean(), hist.std()
                z = (today_val - mean) / std if std > 0 else float("nan")
                records_8.append({
                    "bucket": bucket,
                    "metric": label,
                    "today": today_val,
                    "window": win_label,
                    "mean": mean,
                    "std": std,
                    "z_score": z,
                    "favorable_z": z * sign,
                    "n_hist": len(hist),
                })

    scorecard = pd.DataFrame(records_8)
    print(f"Scorecard dla aukcji {latest_date_sc.date()} (latest): "
          f"{len(scorecard)} records, {scorecard['bucket'].nunique() if not scorecard.empty else 0} buckets")

    if scorecard.empty:
        print("! Brak danych do scorecard - prawdopodobnie dla zadnego bucketu "
              "nie mamy ≥3 historycznych aukcji w oknie 6m.")
        for bucket in sorted(sc_agg["bucket"].dropna().unique()):
            sub = sc_agg[sc_agg.bucket == bucket].sort_values("auction_date")
            n_total = len(sub)
            n_recent = len(sub[sub.auction_date >= (latest_date_sc - pd.Timedelta(days=180))])
            print(f"   {bucket}: total={n_total}, ostatnie 180d={n_recent}, "
                  f"min={sub['auction_date'].min().date() if len(sub) else '-'}, "
                  f"max={sub['auction_date'].max().date() if len(sub) else '-'}")
    else:
        pivoted = scorecard.pivot_table(
            index=["bucket", "metric"],
            columns="window",
            values=["mean", "z_score"],
            aggfunc="first",
        )
        today_vals = scorecard.groupby(["bucket", "metric"])["today"].first()
        out = pd.DataFrame(index=today_vals.index)
        out["dzisiaj"] = today_vals
        if ("mean", "6m") in pivoted.columns:
            out["śr 6m"] = pivoted[("mean", "6m")]
            out["z (6m)"] = pivoted[("z_score", "6m")]
        if ("mean", "12m") in pivoted.columns:
            out["śr 12m"] = pivoted[("mean", "12m")]
            out["z (12m)"] = pivoted[("z_score", "12m")]
        out = out.reset_index()
        out["kierunek"] = out["metric"].map(metric_hint)
        # Reorder: bucket, metric, kierunek, dzisiaj, ...
        col_order = ["bucket", "metric", "kierunek", "dzisiaj"]
        for c in ["śr 6m", "z (6m)", "śr 12m", "z (12m)"]:
            if c in out.columns:
                col_order.append(c)
        out = out[col_order]

        # Row-wise coloring po favorable_z (znak dostosowany do metryki)
        def _color_z_by_favorable(row):
            metric = row["metric"]
            sign = metric_sign.get(metric, +1)
            styles = [""] * len(row)
            for idx, col_name in enumerate(row.index):
                if col_name not in ("z (6m)", "z (12m)"):
                    continue
                z = row[col_name]
                if pd.isna(z):
                    continue
                favorable = z * sign
                a = abs(favorable)
                if a < 1:
                    continue
                if favorable >= 2:
                    styles[idx] = "background-color:#198754;color:white;font-weight:bold"
                elif favorable > 0:
                    styles[idx] = "background-color:#d1e7dd"
                elif favorable > -2:
                    styles[idx] = "background-color:#f8d7da"
                else:
                    styles[idx] = "background-color:#dc3545;color:white;font-weight:bold"
            return styles

        styled = out.style.format({
            "dzisiaj":  "{:+.2f}",
            "śr 6m":    "{:+.2f}",
            "z (6m)":   "{:+.2f}",
            "śr 12m":   "{:+.2f}",
            "z (12m)":  "{:+.2f}",
        }, na_rep="—").hide(axis="index").apply(_color_z_by_favorable, axis=1)
        display(styled)

        # LLM commentary
        _8_lines = []
        for _, r in out.iterrows():
            z6 = r.get("z (6m)")
            z6_str = f"z6m={z6:+.2f}" if pd.notna(z6) else "z6m=n/a"
            _8_lines.append(
                f"  {r.bucket} / {r.metric}: today {r.dzisiaj:+.2f}, "
                f"6m mean {r.get('śr 6m', float('nan')):+.2f}, {z6_str}"
            )
        _summary_8 = (
            f"Scorecard dla aukcji {latest_date_sc.date()} (z-score vs rolling history):\n"
            + "\n".join(_8_lines) +
            "\n\nKierunek 'better': B/C up, Concession down, NK share up."
        )
        display(Markdown(llm_chart_commentary(
            "Chart 13 — Scorecard z z-scores dzisiejszej aukcji",
            "Per coupon bucket (I/OS/Z), porownanie 3 metryk (B/C, Concession, NK share) dzisiaj vs rolling 6m/12m mean+std. Z-score >=1 lub <=-1 sygnalizuje odbieganie od typowego, >=2 lub <=-2 ekstremum.",
            _summary_8,
        )))

## 14. Concession & demand scatters — dzisiejsza aukcja w kontekście całej historii

2×2 scatter plots: każda kropka = jedna aukcja dzienna od 2012, **czerwony diament** = dzisiejsza aukcja. Pozwala zobaczyć czy dzisiaj odbiega od typowej zależności:

- **(górny-lewy) Concession vs Auction size** — czy duże aukcje są droższe (większy tail) niż małe?
- **(górny-prawy) Concession vs Tenor (weighted)** — "duration penalty curve" — typowo długie tenory mają większy concession
- **(dolny-lewy) B/C vs NK share** — czy mocniejsze B/C koreluje z większym udziałem NK (genuine end-user demand)?
- **(dolny-prawy) Auction size vs B/C** — czy MF dostaje słabsze B/C przy większych printach?

Outlier dzisiejszej aukcji vs chmury historii = sygnał czegoś nietypowego do skomentowania.

In [ ]:
# Per-aukcja-dziennie dane do scatters
scat_raw = fetch_view(
    "v_recent_auctions",
    f"?auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,years_to_maturity,sold_total_mln,sold_nc_mln,"
    "demand_total_mln,demand_nc_mln,bid_to_cover,concession_bp",
)
scat_raw["auction_date"] = pd.to_datetime(scat_raw["auction_date"])
for c in ["years_to_maturity", "sold_total_mln", "sold_nc_mln", "demand_total_mln",
          "demand_nc_mln", "bid_to_cover", "concession_bp"]:
    scat_raw[c] = pd.to_numeric(scat_raw[c], errors="coerce")
scat_raw["sold_abs"] = scat_raw["sold_total_mln"].abs()

def _wavg(g, col, w_col):
    mask = g[col].notna() & g[w_col].notna() & (g[w_col] > 0)
    if not mask.any():
        return None
    return (g.loc[mask, col] * g.loc[mask, w_col]).sum() / g.loc[mask, w_col].sum()

agg_scat = scat_raw.groupby("auction_date", group_keys=False).apply(
    lambda g: pd.Series({
        "total_sold_mln":    g["sold_abs"].sum(),
        "total_sold_nk_mln": g["sold_nc_mln"].sum(),
        "total_demand_mln":  g["demand_total_mln"].sum(),
        "total_nk_mln":      g["demand_nc_mln"].sum(),
        "wavg_tenor":        _wavg(g, "years_to_maturity", "sold_abs"),
        "wavg_concession":   _wavg(g, "concession_bp", "sold_abs"),
    })
).reset_index()
agg_scat["bid_to_cover"] = agg_scat["total_demand_mln"] / agg_scat["total_sold_mln"].replace(0, pd.NA)
agg_scat["nk_share_pct"] = agg_scat["total_nk_mln"] / agg_scat["total_demand_mln"].replace(0, pd.NA) * 100

today_date_10 = agg_scat["auction_date"].max()
hist = agg_scat[agg_scat.auction_date < today_date_10]
today = agg_scat[agg_scat.auction_date == today_date_10]

fig = make_subplots(rows=2, cols=2,
                    vertical_spacing=0.13, horizontal_spacing=0.10,
                    subplot_titles=(
                        "Concession vs Auction size",
                        "Concession vs Tenor (ważone)",
                        "B/C vs NK share demand",
                        "Auction size vs B/C",
                    ))

hist_mk = dict(color="lightblue", size=6, opacity=0.55, line=dict(color="steelblue", width=0.3))
today_mk = dict(color="red", size=14, symbol="diamond", line=dict(color="black", width=1))

for r_idx, c_idx, xcol, ycol, xlabel, ylabel in [
    (1, 1, "total_sold_mln", "wavg_concession", "Sold (mln PLN)", "Concession (bp)"),
    (1, 2, "wavg_tenor", "wavg_concession", "Tenor (lata, ważone)", "Concession (bp)"),
    (2, 1, "nk_share_pct", "bid_to_cover", "NK share demand (%)", "B/C"),
    (2, 2, "total_sold_mln", "bid_to_cover", "Sold (mln PLN)", "B/C"),
]:
    fig.add_trace(go.Scatter(
        x=hist[xcol], y=hist[ycol], mode="markers", marker=hist_mk,
        name="historia", showlegend=(r_idx == 1 and c_idx == 1),
    ), row=r_idx, col=c_idx)
    if not today.empty:
        fig.add_trace(go.Scatter(
            x=today[xcol], y=today[ycol], mode="markers", marker=today_mk,
            name="dzisiaj", legendgroup="today",
            showlegend=(r_idx == 1 and c_idx == 1),
        ), row=r_idx, col=c_idx)
    fig.update_xaxes(title_text=xlabel, row=r_idx, col=c_idx)
    fig.update_yaxes(title_text=ylabel, row=r_idx, col=c_idx)

fig.add_hline(y=0, line_dash="dot", line_color="grey", row=1, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="grey", row=1, col=2)
fig.add_hline(y=1, line_dash="dot", line_color="grey", row=2, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="grey", row=2, col=2)

fig.update_layout(
    title=f"Concession & demand scatters — historia + dzisiaj ({today_date_10.date()})",
    template="plotly_white", height=750,
    showlegend=True, legend=dict(orientation="h", y=-0.05),
)
fig.show()

# LLM commentary
if not today.empty:
    t = today.iloc[0]
    _summary_10 = (
        f"Dzisiejsza aukcja {today_date_10.date()}:\n"
        f"  total sold {t.total_sold_mln:,.0f} mln, B/C {t.bid_to_cover:.2f}\n"
        f"  wavg tenor {t.wavg_tenor:.1f}Y, wavg conc {t.wavg_concession:+.1f}bp\n"
        f"  NK share {t.nk_share_pct:.1f}%\n\n"
        f"Historical context (n={len(hist)} prior auctions):\n"
        f"  Median sold: {hist.total_sold_mln.median():,.0f} mln\n"
        f"  Median B/C: {hist.bid_to_cover.median():.2f}\n"
        f"  Median conc: {hist.wavg_concession.median():+.1f}bp\n"
        f"  Median tenor: {hist.wavg_tenor.median():.1f}Y\n"
        f"  Median NK share: {hist.nk_share_pct.median():.1f}%"
    )
else:
    _summary_10 = "Brak dzisiejszej aukcji w danych."
display(Markdown(llm_chart_commentary(
    "Chart 14 — Concession & demand scatters",
    "2x2 scatter plots: concession vs size, vs tenor, B/C vs NK share, size vs B/C. Light blue = historia, czerwony diament = dzisiaj. Pokazuje czy dzisiejszy print odbiega od typowej zaleznosci - sygnal czegos nietypowego.",
    _summary_10,
)))

## 15. NK (non-competitive) bidding — trend, udział i fill rate

NK to polski odpowiednik US "indirect bidders" — wskaźnik genuine end-user demand vs dealer flippingu. NK award jest cap-owane per dealer (jako % od competitive award), więc fully utilised NK pool sygnalizuje strong real-money demand.

3 panele:
- **Górny** — stacked bar: competitive demand vs NK demand per aukcja
- **Środkowy** — NK share % popytu (kropki + MA(12) ±1 SD band)
- **Dolny** — **NK fill rate %** = `sold_NK / demand_NK` — jaka część popytu NK została zaspokojona:
  - **100%** = NK pool nie był binding constraint (cała chęć zaspokojona)
  - **< 100%** = NK rationowane (oversubscribed, cap binding) — silny end-user demand
  - dropy poniżej rolling band = exceptional NK appetite

In [ ]:
import numpy as np

day_nk = agg_scat[[
    "auction_date", "total_demand_mln", "total_nk_mln",
    "total_sold_nk_mln", "nk_share_pct"
]].copy().sort_values("auction_date").reset_index(drop=True)
day_nk["comp_demand"] = day_nk["total_demand_mln"] - day_nk["total_nk_mln"]
day_nk["nk_fill_pct"] = (
    day_nk["total_sold_nk_mln"] / day_nk["total_nk_mln"].replace(0, np.nan) * 100
).clip(upper=100).astype(float)
day_nk["nk_share_pct"] = pd.to_numeric(day_nk["nk_share_pct"], errors="coerce")

WIN_11 = 12
day_nk["roll_share"] = day_nk["nk_share_pct"].rolling(WIN_11, min_periods=4).mean()
day_nk["roll_share_std"] = day_nk["nk_share_pct"].rolling(WIN_11, min_periods=4).std()
day_nk["roll_fill"] = day_nk["nk_fill_pct"].rolling(WIN_11, min_periods=4).mean()
day_nk["roll_fill_std"] = day_nk["nk_fill_pct"].rolling(WIN_11, min_periods=4).std()

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    row_heights=[0.40, 0.30, 0.30],
    subplot_titles=(
        "Demand stacked: competitive (niebieski) + NK (pomarańczowy)",
        f"NK share % popytu — kropki + MA({WIN_11}) ±1 SD",
        f"NK fill rate % = sold_NK / demand_NK — <100% = rationowane (oversubscribed)",
    ),
)

fig.add_trace(go.Bar(
    x=day_nk["auction_date"], y=day_nk["comp_demand"],
    name="Competitive demand", marker_color="lightsteelblue",
    hovertemplate="%{x|%Y-%m-%d}<br>comp: %{y:,.0f} mln<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=day_nk["auction_date"], y=day_nk["total_nk_mln"],
    name="NK demand", marker_color="darkorange",
    hovertemplate="%{x|%Y-%m-%d}<br>NK: %{y:,.0f} mln<extra></extra>",
), row=1, col=1)

# Row 2: NK share + band + ma
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"] + day_nk["roll_share_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"] - day_nk["roll_share_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(255,165,0,0.18)",
    name="±1 SD (share)", hoverinfo="skip",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["nk_share_pct"],
    mode="markers", marker=dict(color="darkorange", size=4, opacity=0.55),
    name="NK share %",
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra></extra>",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"],
    mode="lines", line=dict(color="darkorange", width=2.5),
    name=f"MA({WIN_11}) share",
), row=2, col=1)

# Row 3: NK fill rate
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"] + day_nk["roll_fill_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"] - day_nk["roll_fill_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(150,80,200,0.18)",
    name="±1 SD (fill)", hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["nk_fill_pct"],
    mode="markers", marker=dict(color="#8B4FBA", size=4, opacity=0.55),
    name="NK fill %",
    hovertemplate="%{x|%Y-%m-%d}<br>fill: %{y:.1f}%<extra></extra>",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"],
    mode="lines", line=dict(color="#8B4FBA", width=2.5),
    name=f"MA({WIN_11}) fill",
), row=3, col=1)
fig.add_hline(y=100, line_dash="dot", line_color="grey", row=3, col=1,
              annotation_text="100% = brak rationowania", annotation_position="top right")

fig.update_yaxes(title_text="mln PLN", row=1, col=1)
fig.update_yaxes(title_text="NK share (%)", row=2, col=1)
fig.update_yaxes(title_text="NK fill rate (%)", row=3, col=1, range=[0, 105])
fig.update_layout(
    title="NK (non-competitive) bidding — historia od 2012",
    template="plotly_white", barmode="stack",
    height=850, hovermode="x unified",
    legend=dict(orientation="h", y=-0.08),
)
fig.show()

if not day_nk.empty:
    latest = day_nk.iloc[-1]
    print(f"\nLatest auction {latest['auction_date'].date()}:")
    print(f"  Total demand: {latest['total_demand_mln']:,.0f} mln "
          f"(comp: {latest['comp_demand']:,.0f}, NK: {latest['total_nk_mln']:,.0f})")
    print(f"  NK share:    {latest['nk_share_pct']:.1f}% "
          f"(MA{WIN_11}: {latest['roll_share']:.1f}% ±{latest['roll_share_std']:.1f})")
    if pd.notna(latest["nk_fill_pct"]):
        print(f"  NK fill rate: {latest['nk_fill_pct']:.1f}% "
              f"(sold {latest['total_sold_nk_mln']:,.0f} / demand {latest['total_nk_mln']:,.0f}) "
              f"— MA{WIN_11}: {latest['roll_fill']:.1f}%")
    else:
        print(f"  NK fill rate: brak (demand_nk = 0)")

# LLM commentary
_11_latest = day_nk.iloc[-1] if len(day_nk) else None
_summary_11 = "no data"
if _11_latest is not None:
    _summary_11 = (
        f"Latest auction ({_11_latest.auction_date.date()}):\n"
        f"  Total demand: {_11_latest.total_demand_mln:,.0f} mln "
        f"(comp {_11_latest.comp_demand:,.0f} + NK {_11_latest.total_nk_mln:,.0f})\n"
        f"  NK share: {_11_latest.nk_share_pct:.1f}% (MA{WIN_11}: {_11_latest.roll_share:.1f}%)\n"
        f"  NK fill rate: {_11_latest.nk_fill_pct:.1f}% (MA{WIN_11}: {_11_latest.roll_fill:.1f}%)\n"
        f"  Fill <100% = NK pool rationowany (oversubscribed, end-user demand >cap)"
    )
display(Markdown(llm_chart_commentary(
    "Chart 15 — NK (non-competitive) bidding",
    "3 panele: stacked competitive+NK demand, NK share % popytu z MA12, NK fill rate (sold/demand). Fill <100% = NK pool oversubscribed (silny real-money demand, cap binding); 100% = caly NK popyt zaspokojony. Polski odpowiednik US 'indirect bidders'.",
    _summary_11,
)))

## 16. Funding pace tracker — YTD vs MF rocznego planu

Cumulative gross + net issuance YTD vs MF rocznego planu (z dokumentu strategii długu publicznego). Annotacja na ostatnim dniu pokazuje **% planu vs % kalendarza** — jeśli % planu < % kalendarza, MF jest "behind schedule" (zostało więcej do uplasowania w mniejszej części roku).

- **Niebieska linia**: cumulative gross — sumarycznie sprzedano w aukcjach POLGB (AS+AU+SD+AZ-sale)
- **Zielona linia**: net = gross - cumulative redemptions YTD (wykupy obligacji hurtowych z tego roku)
- **Szara dashed**: linear pace reference — gdzie byłoby gdyby pace był równomierny przez cały rok

Targety per rok w `MF_GROSS_TARGET_BLN_PLN`. Update gdy MF publikuje nowy plan (zwykle październik–grudzień poprzedniego roku).

In [ ]:
# MF planowana emisja brutto POLGB HURTOWE w bln PLN.
MF_GROSS_TARGET_BLN_PLN = {
    2024: 380.0,
    2025: 415.0,
    2026: 348.0,
}

current_year = TODAY.year
target = MF_GROSS_TARGET_BLN_PLN.get(current_year)
year_start = pd.Timestamp(year=current_year, month=1, day=1)
year_end = pd.Timestamp(year=current_year, month=12, day=31)

# Gross YTD
gross_raw = fetch_view(
    "bond_auctions",
    f"?type_tx=eq.S&type_op=in.(AS,AU,SD,AZ)"
    f"&auction_date=gte.{year_start.strftime('%Y-%m-%d')}"
    "&select=auction_date,sold_total_mln",
)
gross_raw["auction_date"] = pd.to_datetime(gross_raw["auction_date"])
gross_raw["sold_total_mln"] = pd.to_numeric(gross_raw["sold_total_mln"], errors="coerce")
gross_daily = gross_raw.groupby("auction_date")["sold_total_mln"].sum().sort_index()

# Redemptions YTD
redemp_raw = fetch_view(
    "bond_outstanding",
    f"?op_type=eq.redemption&change_date=gte.{year_start.strftime('%Y-%m-%d')}"
    "&select=change_date,delta_mln_pln",
)
redemp_raw["change_date"] = pd.to_datetime(redemp_raw["change_date"])
redemp_raw["delta_mln_pln"] = pd.to_numeric(redemp_raw["delta_mln_pln"], errors="coerce")
redemp_daily = (-redemp_raw.groupby("change_date")["delta_mln_pln"].sum()).sort_index()

all_dates = pd.date_range(year_start, min(TODAY, year_end), freq="D")
gross_cumul = (gross_daily.reindex(all_dates).fillna(0).cumsum()) / 1000
redemp_cumul = (redemp_daily.reindex(all_dates).fillna(0).cumsum()) / 1000
net_cumul = gross_cumul - redemp_cumul

linear_pace = None
if target:
    days_in_year = (year_end - year_start).days + 1
    linear_pace = pd.Series(
        [target * (i + 1) / days_in_year for i in range(len(all_dates))],
        index=all_dates,
    )

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gross_cumul.index, y=gross_cumul,
    mode="lines", line=dict(color="navy", width=2.5),
    name="Gross cumulative YTD",
    hovertemplate="%{x|%Y-%m-%d}<br>gross: %{y:.1f} bln PLN<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=net_cumul.index, y=net_cumul,
    mode="lines", line=dict(color="darkgreen", width=2.5),
    name="Net (gross - wykupy) YTD",
    hovertemplate="%{x|%Y-%m-%d}<br>net: %{y:.1f} bln PLN<extra></extra>",
))
if linear_pace is not None:
    fig.add_trace(go.Scatter(
        x=linear_pace.index, y=linear_pace,
        mode="lines", line=dict(color="grey", dash="dash", width=1.5),
        name=f"Linear pace → {target:.0f} bln PLN",
        hoverinfo="skip",
    ))
    fig.add_hline(y=target, line_dash="dot", line_color="grey",
                  annotation_text=f"Target {target:.0f} bln PLN",
                  annotation_position="top right")

today_gross = gross_cumul.iloc[-1] if len(gross_cumul) else 0
today_net = net_cumul.iloc[-1] if len(net_cumul) else 0
pct_cal = ((TODAY - year_start).days + 1) / ((year_end - year_start).days + 1) * 100
if target:
    pct_plan = today_gross / target * 100
    gap = pct_plan - pct_cal
    gap_str = (f"<b>{today_gross:.0f} bln gross</b> ({pct_plan:.1f}% planu) "
               f"vs {pct_cal:.1f}% kalendarza "
               f"= <b>{'+' if gap >= 0 else ''}{gap:.1f}pp {'ahead' if gap >= 0 else 'behind'}</b>")
else:
    gap_str = f"<b>{today_gross:.0f} bln gross</b> ({pct_cal:.1f}% kalendarza)<br>brak target dla {current_year}"

fig.add_annotation(
    x=all_dates[-1], y=today_gross,
    text=gap_str, showarrow=True, arrowhead=2, ax=-100, ay=-60,
    bgcolor="white", bordercolor="black", borderwidth=1,
)

fig.update_layout(
    title=f"Funding pace {current_year} — POLGB hurtowe vs MF roczny plan",
    xaxis_title="Data", yaxis_title="bln PLN",
    template="plotly_white", height=550,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

print(f"\n{current_year} progress:")
print(f"  Gross YTD: {today_gross:.1f} bln PLN")
print(f"  Net YTD:   {today_net:.1f} bln PLN")
print(f"  Calendar:  {pct_cal:.1f}%")
if target:
    print(f"  Plan:      {pct_plan:.1f}% of {target:.0f} bln PLN target")
    print(f"  Gap:       {gap:+.1f}pp ({'ahead' if gap >= 0 else 'behind'} schedule)")
else:
    print(f"  ! Brak targetu w MF_GROSS_TARGET_BLN_PLN dla {current_year}")

# LLM commentary
_summary_9 = (
    f"{current_year} YTD progress (snapshot {TODAY.date()}):\n"
    f"  Gross issued: {today_gross:.1f} bln PLN ({pct_plan:.1f}% of target {target:.0f} bln)\n"
    f"  Net (gross - wykupy): {today_net:.1f} bln PLN\n"
    f"  Calendar elapsed: {pct_cal:.1f}%\n"
    f"  Gap (plan vs kalendarz): {gap:+.1f}pp ({'ahead' if gap >= 0 else 'behind'} schedule)"
) if target else (
    f"{current_year} YTD: gross {today_gross:.1f} bln, net {today_net:.1f} bln; "
    f"brak targetu zdefiniowanego dla tego roku."
)
display(Markdown(llm_chart_commentary(
    "Chart 16 — Funding pace tracker",
    "Cumulative gross + net POLGB hurtowe issuance YTD vs roczny target z MF Strategii. Linear pace = gdzie powinno byc gdyby tempo bylo rownomierne. Annotacja w trakcie roku pokazuje 'ahead/behind schedule' - sygnal czy MF jest pod presja czy ma comfort.",
    _summary_9,
)))

---

# 🤖 Raport finalny — analiza dzisiejszej aukcji

Synteza wszystkich poprzednich wykresów + kontekst makroekonomiczny → profesjonalny raport sell-side w jezyku polskim, generowany przez Claude Opus 4.8.

In [ ]:
# Zbierz kluczowe dane ze wszystkich sekcji - aggregowane w jeden context string
# ktory LLM dostaje jako input do finalnego raportu.
# Najpierw macro snapshot (PL10Y/DE10Y/US10Y/FX/WIBOR/funding pace) zeby LLM
# mial GROUNDED kontekst makro zamiast halucynowac z treningu.

_macro_ctx = fetch_macro_snapshot()
_macro_block = format_macro_block(_macro_ctx)

_final_ctx_parts = [
    f"DZISIEJSZA DATA: {TODAY.date()}",
    "",
    "=== PORTFOLIO TODAY (chart 1) ===",
    f"Total outstanding: {df1.iloc[-1].total_outstanding_mln_pln/1000:.0f} bln PLN",
    f"Mod Duration: {df1.iloc[-1].portfolio_mod_duration:.2f}Y, "
    f"Mac Duration: {df1.iloc[-1].portfolio_mac_duration:.2f}Y, "
    f"ATM: {df1.iloc[-1].portfolio_atm:.2f}Y, "
    f"ATR: {df1.iloc[-1].portfolio_atr:.2f}Y",
    "",
    "=== STRUKTURA DLUGU (chart 3b) ===",
] + [f"  {b}: {piv_pct.iloc[-1][b]:.1f}% ({piv_buckets.iloc[-1][b]/1000:.0f} bln)"
     for b in piv_pct.columns] + [
    "",
    "=== OSTATNIA AUKCJA (chart 4a/4c) ===",
    f"Data: {latest_date}",
    f"Total sold: {total_sold:,.0f} mln, total demand: {total_demand:,.0f} mln, "
    f"overall B/C: {overall_bc:.2f}",
    f"Serie ({len(df_last)}): " + ", ".join(df_last["seria"].tolist()),
    "",
    "Per seria detail:",
] + [f"  {r.seria} ({r.type_op}/{r.years_to_maturity}Y/{r.coupon_kind}): "
     f"sold {r.sold_total_mln:,.0f}, B/C {r.bid_to_cover:.2f}, "
     f"yld {r.yield_avg:.3f}%, conc {r.concession_bp:+.1f}bp, "
     f"NK {r.nc_share_demand*100:.1f}%"
     for _, r in df_last.iterrows()] + [
    "",
    "=== TAIL / CONCESSION CONTEXT (chart 7) ===",
    f"Latest concession: {day_conc.iloc[-1].concession_bp:+.1f} bp "
    f"(MA6 {day_conc.iloc[-1].roll_mean:+.1f} ±{day_conc.iloc[-1].roll_std:.1f})",
    "",
    "=== PORTFOLIO IMPACT DZISIAJ (chart 5) ===",
    f"Δ Mod: {df5.iloc[-1].impact_portfolio_mod_duration:+.4f}Y, "
    f"Δ Mac: {df5.iloc[-1].impact_portfolio_mac_duration:+.4f}Y, "
    f"Δ ATM: {df5.iloc[-1].impact_portfolio_atm:+.4f}Y, "
    f"Δ ATR: {df5.iloc[-1].impact_portfolio_atr:+.4f}Y",
    "",
    "=== NK BIDDING (chart 11) ===",
    f"Latest NK share: {day_nk.iloc[-1].nk_share_pct:.1f}% (MA12 {day_nk.iloc[-1].roll_share:.1f}%)",
    f"Latest NK fill rate: {day_nk.iloc[-1].nk_fill_pct:.1f}% "
    f"(MA12 {day_nk.iloc[-1].roll_fill:.1f}%) - <100% = oversubscribed",
    "",
    "=== FUNDING PACE (chart 9) ===",
    f"YTD gross: {today_gross:.0f} bln, net: {today_net:.0f} bln",
    f"Plan {current_year}: {target:.0f} bln" if target else f"Brak targetu {current_year}",
    f"% planu: {pct_plan:.1f}% vs % kalendarza: {pct_cal:.1f}% = "
    f"{gap:+.1f}pp ({'ahead' if gap >= 0 else 'behind'})" if target else "",
    "",
    "=== MATURITY LADDER (chart 12) ===",
    f"Total outstanding: {total_outstanding/1000:.0f} bln, WAM {wam_yrs:.2f}Y",
    f"Peak year: {peak_year} ({peak_val/1000:.1f} bln)",
    f"Next 12M maturities: {mat_12m/1000:.1f} bln "
    f"({100*mat_12m/total_outstanding:.1f}% portfela)",
    f"Matured YTD: {matured_ytd_total/1000:.1f} bln",
]

_final_context = _macro_block + "\n".join(p for p in _final_ctx_parts if p is not None)

display(Markdown(llm_final_report(_final_context)))
print(f"\n_Raport wygenerowany przez {LLM_MODEL} o {pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d %H:%M UTC')}_")

---

**Tip:** wykresy plotly są interaktywne — najedź myszą żeby zobaczyć wartości, zaznacz prostokąt żeby przybliżyć, podwójny klik żeby zresetować. Trace w legendzie można wyłączać klikiem.